# LeetCode 302 — Smallest Rectangle Enclosing Black Pixels (Hard) bằng Python

> **Lưu ý: đây là bài Premium.** Trên LeetCode, bài 302 bị khóa — cần tài khoản LeetCode Premium mới xem được đề và nộp bài. Đề bài, ví dụ, constraints và chữ ký hàm dưới đây được chép lại theo đúng bản LeetCode, để bạn học trọn vẹn ngay cả khi không có Premium.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minArea(self, image: List[List[str]], x: int, y: int) -> int:

```

Đề cho một "bức ảnh" đen trắng dạng lưới. Các pixel đen dính liền nhau thành **một khối**. Hãy tìm hình chữ nhật nhỏ nhất bao trọn khối đen đó. Cách "ngây thơ" — nhìn hết mọi ô — đã cho đáp án đúng, nhưng đề **đòi nhanh hơn `O(mn)`**, tức là không được nhìn hết ảnh! Bí quyết nằm ở một quan sát đẹp: vì khối đen liền mạch, "bóng" của nó in xuống trục ngang (và trục dọc) là **một đoạn liền không có lỗ**. Nhờ vậy ta tìm được hai đầu đoạn bằng **binary search** (tìm kiếm nhị phân) — dù ảnh **không hề được sắp xếp**. Đây là ví dụ kinh điển của "binary search trên một câu hỏi Có/Không đơn điệu", một kỹ thuật dùng được ở rất nhiều bài khác.

---

## Sau notebook này, bạn sẽ nắm được

1. **Lưới 2D** trong Python (list lồng list), chỉ số `image[r][c]`, quy ước `x` = hàng, `y` = cột; vẽ lưới bằng ASCII để "nhìn thấy" ảnh.
2. **Bounding box** (hình chữ nhật bao nhỏ nhất) = min/max của hàng và cột; công thức diện tích và lỗi "lệch một" (`+ 1`).
3. **Liên thông 4 hướng** (4-directional connectivity): hàng xóm, kiểm tra biên, thế nào là "một vùng".
4. **Cách 1 — Quét toàn bộ ảnh** `O(mn)`: dễ nhất, dùng làm **trọng tài** để kiểm tra các cách khác.
5. **Cách 2 — BFS/DFS từ `(x, y)`** `O(B)`: học hàng đợi `deque`, tập `visited`, "vết dầu loang" (flood fill).
6. **Cách 3 — Binary search trên hình chiếu (cách chính)** `O(m log n + n log m)`: vì sao **liên thông ⇒ đơn điệu**, phản ví dụ khi không liên thông, hai hàm `has_black_in_col / has_black_in_row`, trace kỹ từng bước.
7. **Cách 4 — Một hàm binary search tổng quát** `first_true(lo, hi, pred)` dùng chung cho cả 4 biên; học `lambda` và truyền hàm làm tham số.
8. Code nộp LeetCode, kiểm thử chéo trên hàng nghìn ảnh ngẫu nhiên liên thông, lỗi hay gặp, mẹo nhận dạng, biến thể và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

You are given an `m x n` binary matrix `image` where `0` represents a white pixel and `1` represents a black pixel.

The black pixels are connected (i.e., there is only one black region). Pixels are connected horizontally and vertically.

Given two integers `x` and `y` that represents the location of one of the black pixels, return *the area of the smallest (axis-aligned) rectangle that encloses all black pixels*.

You must write an algorithm with less than `O(mn)` runtime complexity.

**Example 1**

```text
Input:  image = [["0","0","1","0"],["0","1","1","0"],["0","1","0","0"]], x = 0, y = 2
Output: 6
```

**Example 2**

```text
Input:  image = [["1"]], x = 0, y = 0
Output: 1
```

**Constraints**

- `m == image.length`
- `n == image[i].length`
- `1 <= m, n <= 100`
- `image[i][j]` is either `'0'` or `'1'`.
- `0 <= x < m`
- `0 <= y < n`
- `image[x][y] == '1'`.
- The black pixels in the `image` only form **one component**.

---

**Dịch nhanh và những điều cần để ý:**

- `image` là lưới `m` hàng × `n` cột. Mỗi ô là **chuỗi** `'0'` (trắng) hoặc `'1'` (đen) — chuỗi một ký tự, **không phải số** `0/1`.
- Các pixel đen **liên thông** (connected): từ một ô đen bất kỳ có thể đi tới mọi ô đen khác, mỗi bước sang một ô đen kề **trên / dưới / trái / phải** (không đi chéo). Nói cách khác, ảnh chỉ có **đúng một vùng đen**.
- `(x, y)` là vị trí một pixel đen: **`x` là chỉ số hàng, `y` là chỉ số cột** (vì `image[x][y] == '1'`). Ngược với thói quen "x nằm ngang, y thẳng đứng" của môn toán — rất dễ nhầm!
- Trả về **diện tích** (số ô) của hình chữ nhật nhỏ nhất có cạnh song song với lưới (axis-aligned) chứa **mọi** pixel đen. Hình chữ nhật được phép chứa cả ô trắng.
- Yêu cầu thuật toán **nhanh hơn `O(mn)`**: về lý thuyết là không được đọc hết mọi ô. Với `m, n <= 100`, lời giải `O(mn)` thường vẫn được chấp nhận (Accepted), nhưng đề đã nêu rõ yêu cầu và người phỏng vấn sẽ hỏi đúng điểm này.
- Example 1: các ô đen ở `(0,2), (1,1), (1,2), (2,1)` → chiếm hàng `0..2` (3 hàng) và cột `1..2` (2 cột) → diện tích `3 × 2 = 6`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

image1 = [["0", "0", "1", "0"],
          ["0", "1", "1", "0"],
          ["0", "1", "0", "0"]]
x1, y1 = 0, 2

image2 = [["1"]]
x2, y2 = 0, 0

print("Example 1:", image1, "x =", x1, "y =", y1, "-> mong đợi 6")
print("Example 2:", image2, "x =", x2, "y =", y2, "-> mong đợi 1")
print("image1[x1][y1] =", repr(image1[x1][y1]), "(đúng là pixel đen)")

Example 1: [['0', '0', '1', '0'], ['0', '1', '1', '0'], ['0', '1', '0', '0']] x = 0 y = 2 -> mong đợi 6
Example 2: [['1']] x = 0 y = 0 -> mong đợi 1
image1[x1][y1] = '1' (đúng là pixel đen)


## 2. Nền tảng: lưới 2D (grid) trong Python

### 2.1. List lồng list

Một lưới `m × n` được lưu dưới dạng **list của các list**: `image` là list gồm `m` phần tử, mỗi phần tử `image[r]` là **một hàng** — bản thân nó lại là một list gồm `n` ô.

- `len(image)` = số hàng `m`; `len(image[0])` = số cột `n` (độ dài của một hàng).
- `image[r]` = cả hàng `r` (một list).
- `image[r][c]` = ô ở **hàng `r`, cột `c`**. Đọc từ trái sang: "lấy hàng `r` trước, rồi lấy ô thứ `c` trong hàng đó".

Example 1 kèm chỉ số:

```text
          cột c →   0    1    2    3
hàng r = 0       [ "0", "0", "1", "0" ]
hàng r = 1       [ "0", "1", "1", "0" ]
hàng r = 2       [ "0", "1", "0", "0" ]
```

`image1[0][2]` → hàng 0, cột 2 → `"1"`. Còn `image1[2][0]` → hàng 2, cột 0 → `"0"`. Đổi chỗ hai chỉ số là ra ô khác hẳn!

Quy ước dùng trong cả notebook: `r` (row) = chỉ số hàng, tăng **từ trên xuống**; `c` (column) = chỉ số cột, tăng **từ trái sang**. Đề gọi pixel xuất phát là `(x, y)` thì `x` đóng vai `r`, `y` đóng vai `c`.

### 2.2. Cẩn thận: ô là chuỗi `'1'`, không phải số `1`

Trong Python, `'1'` (chuỗi) và `1` (số nguyên) là **hai thứ khác nhau**: `'1' == 1` cho `False`. Nếu lỡ viết `if image[r][c] == 1:` thì điều kiện **không bao giờ đúng** → chương trình tưởng ảnh toàn trắng. Luôn so với `'1'`.

Hai công cụ có sẵn rất tiện khi làm việc với **một hàng** `row`:

- `'1' in row` → `True` nếu hàng có ít nhất một ô đen. Toán tử `in` duyệt list từ trái sang, **gặp là dừng ngay**.
- `row.count('1')` → đếm số ô đen trong hàng.

In [2]:
m, n = len(image1), len(image1[0])
print("m (số hàng) =", m, "| n (số cột) =", n)
print("image1[0]    =", image1[0], "<- cả hàng 0")
print("image1[0][2] =", image1[0][2], "<- hàng 0, cột 2")
print("image1[2][0] =", image1[2][0], "<- hàng 2, cột 0")

for r in range(m):                      # duyệt từng ô: hàng ngoài, cột trong
    for c in range(n):
        print(f"({r},{c})={image1[r][c]}", end="  ")
    print()                             # xuống dòng sau mỗi hàng

print("'1' == 1   ->", '1' == 1)
print("'1' == '1' ->", '1' == '1')
print("'1' in hàng 1 ->", '1' in image1[1], "| số ô đen hàng 1 ->", image1[1].count('1'))
print("'1' in ['0', '0'] ->", '1' in ['0', '0'])

m (số hàng) = 3 | n (số cột) = 4
image1[0]    = ['0', '0', '1', '0'] <- cả hàng 0
image1[0][2] = 1 <- hàng 0, cột 2
image1[2][0] = 0 <- hàng 2, cột 0
(0,0)=0  (0,1)=0  (0,2)=1  (0,3)=0  
(1,0)=0  (1,1)=1  (1,2)=1  (1,3)=0  
(2,0)=0  (2,1)=1  (2,2)=0  (2,3)=0  
'1' == 1   -> False
'1' == '1' -> True
'1' in hàng 1 -> True | số ô đen hàng 1 -> 2
'1' in ['0', '0'] -> False


### 2.3. Viết ảnh cho dễ đọc và vẽ ảnh bằng ASCII

Gõ `[["0","0","1","0"], ...]` rất mỏi tay và khó nhìn. Ta viết hàm `parse_image` đổi một list các chuỗi kiểu `"..#."` (dấu `#` = đen, `.` = trắng) thành đúng định dạng của đề. Hàm dùng **list comprehension** — cú pháp `[biểu_thức for phần_tử in dãy]` để tạo list gọn trong một dòng. Ví dụ `[ch for ch in "abc"]` cho `['a', 'b', 'c']`; `['1' if ch == '#' else '0' for ch in "..#."]` cho `['0', '0', '1', '0']`.

Hàm `show_grid` thì làm ngược lại: in ảnh ra màn hình với `#` = đen, `.` = trắng, có số hàng (bên trái) và số cột (bên trên). Hai tham số tùy chọn:

- `start=(x, y)`: in ô xuất phát bằng ký tự `@` cho dễ thấy.
- `box=(top, left, bottom, right)`: các ô **trắng nằm trong** hình chữ nhật này được in bằng `+` → nhìn rõ hình chữ nhật bao.

Ta cũng định nghĩa luôn một ảnh lớn hơn `image_big` (6 × 10), hình dạng "uốn khúc", để dùng xuyên suốt notebook. Trên ảnh này các thuật toán sẽ đi qua nhiều nhánh thú vị hơn Example 1. Bạn không cần thuộc code vẽ — chỉ cần đọc được hình nó in ra.

In [3]:
def parse_image(rows):
    """Đổi ['..#.', '.##.'] thành [['0','0','1','0'], ['0','1','1','0']]."""
    return [['1' if ch == '#' else '0' for ch in row] for row in rows]


def show_grid(image, start=None, box=None, title=None):
    m, n = len(image), len(image[0])
    if title:
        print(title)
    print("     " + " ".join(str(c % 10) for c in range(n)))   # số cột (chỉ in chữ số cuối)
    for r in range(m):
        cells = []
        for c in range(n):
            if start == (r, c):
                ch = "@"
            elif image[r][c] == '1':
                ch = "#"
            elif box and box[0] <= r <= box[2] and box[1] <= c <= box[3]:
                ch = "+"
            else:
                ch = "."
            cells.append(ch)
        print(f"{r:>3}  " + " ".join(cells))


print("parse_image cho đúng Example 1?", parse_image(["..#.", ".##.", ".#.."]) == image1)
show_grid(image1, start=(x1, y1), title="Example 1 (@ = pixel xuất phát (x, y) = (0, 2)):")
print()
image_big = parse_image([
    "..........",
    "...###....",
    "...#.###..",
    ".###...#..",
    ".......#..",
    "..........",
])
x_big, y_big = 2, 5
show_grid(image_big, start=(x_big, y_big), title="image_big (6 x 10), (x, y) = (2, 5):")

parse_image cho đúng Example 1? True
Example 1 (@ = pixel xuất phát (x, y) = (0, 2)):
     0 1 2 3
  0  . . @ .
  1  . # # .
  2  . # . .

image_big (6 x 10), (x, y) = (2, 5):
     0 1 2 3 4 5 6 7 8 9
  0  . . . . . . . . . .
  1  . . . # # # . . . .
  2  . . . # . @ # # . .
  3  . # # # . . . # . .
  4  . . . . . . . # . .
  5  . . . . . . . . . .


## 3. Nền tảng: hình chữ nhật bao nhỏ nhất (bounding box)

Hãy tưởng tượng bạn cắt một tấm bìa hình chữ nhật (cạnh song song với lưới) sao cho **che kín mọi ô đen**, và muốn tấm bìa **nhỏ nhất có thể**. Tấm bìa đó gọi là **bounding box** (hộp bao). Nó được xác định bởi 4 con số:

| biên | ý nghĩa | công thức |
|---|---|---|
| `top` | hàng trên cùng có ô đen | `min` của các chỉ số hàng `r` của ô đen |
| `bottom` | hàng dưới cùng có ô đen | `max` của các `r` |
| `left` | cột trái nhất có ô đen | `min` của các chỉ số cột `c` của ô đen |
| `right` | cột phải nhất có ô đen | `max` của các `c` |

**Tại sao đây là hình nhỏ nhất?** Nếu tấm bìa bắt đầu từ hàng `top + 1` thì ô đen ở hàng `top` bị lòi ra ngoài; nếu bắt đầu từ hàng `top - 1` thì thừa một hàng. Mỗi biên đều "vừa khít", nên không thể nhỏ hơn.

**Quan sát quan trọng:** 4 biên **độc lập** với nhau. `top/bottom` chỉ phụ thuộc vào chỉ số **hàng** của các ô đen; `left/right` chỉ phụ thuộc vào chỉ số **cột**. Ý này sẽ là chìa khóa của cách tối ưu ở mục 8.

Tính tay Example 1: ô đen là `(0,2), (1,1), (1,2), (2,1)`.

- Các chỉ số hàng: `0, 1, 1, 2` → `top = 0`, `bottom = 2`.
- Các chỉ số cột: `2, 1, 2, 1` → `left = 1`, `right = 2`.

### 3.1. Công thức diện tích — đừng quên `+ 1`

Hàng từ `top = 0` tới `bottom = 2` gồm các hàng `0, 1, 2` → **3 hàng**, không phải `2 - 0 = 2`. Tổng quát: số các số nguyên từ `a` tới `b` (tính cả hai đầu) là `b - a + 1`. Đây là lỗi kinh điển **"lệch một"** (off-by-one), còn gọi là "lỗi hàng rào": hàng rào dài 10 mét, cứ mỗi mét cắm một cọc thì cần **11** cọc chứ không phải 10.

```text
height = bottom - top + 1     # số hàng
width  = right - left + 1     # số cột
area   = height * width
```

Example 1: `height = 2 - 0 + 1 = 3`, `width = 2 - 1 + 1 = 2`, `area = 3 * 2 = 6` ✔.

Để ý: hình chữ nhật chứa cả ô **trắng** (ô `(0,1)` và `(2,2)` trong Example 1). Đề hỏi diện tích **hình chữ nhật**, không phải số ô đen (số ô đen là 4).

In [4]:
black = [(r, c) for r in range(len(image1)) for c in range(len(image1[0])) if image1[r][c] == '1']
print("Các ô đen:", black)

rows = [r for r, c in black]
cols = [c for r, c in black]
top, bottom = min(rows), max(rows)
left, right = min(cols), max(cols)
print("chỉ số hàng:", rows, "-> top =", top, ", bottom =", bottom)
print("chỉ số cột :", cols, "-> left =", left, ", right =", right)

print("các hàng từ 0 tới 2:", list(range(top, bottom + 1)), "-> có", bottom - top + 1, "hàng")
print("các cột từ 1 tới 2 :", list(range(left, right + 1)), "-> có", right - left + 1, "cột")
height, width = bottom - top + 1, right - left + 1
print(f"height = {height}, width = {width}, area = {height * width}")
show_grid(image1, box=(top, left, bottom, right), title="Hộp bao của Example 1 (+ = ô trắng nằm trong hộp):")

Các ô đen: [(0, 2), (1, 1), (1, 2), (2, 1)]
chỉ số hàng: [0, 1, 1, 2] -> top = 0 , bottom = 2
chỉ số cột : [2, 1, 2, 1] -> left = 1 , right = 2
các hàng từ 0 tới 2: [0, 1, 2] -> có 3 hàng
các cột từ 1 tới 2 : [1, 2] -> có 2 cột
height = 3, width = 2, area = 6
Hộp bao của Example 1 (+ = ô trắng nằm trong hộp):
     0 1 2 3
  0  . + # .
  1  . # # .
  2  . # + .


## 4. Nền tảng: liên thông 4 hướng (4-directional connectivity)

Hai ô gọi là **kề nhau** (adjacent — hàng xóm) nếu chung một **cạnh**. Ô `(r, c)` có tối đa 4 hàng xóm:

```text
               (r-1, c)
                  ↑
   (r, c-1)  ←  (r, c)  →  (r, c+1)
                  ↓
               (r+1, c)
```

Ô chéo như `(r-1, c+1)` chỉ chung một **góc** → **không** kề (theo đề: "connected horizontally and vertically").

Ta viết 4 hướng thành list `DIRS` gồm các cặp `(dr, dc)` (độ lệch hàng, độ lệch cột); hàng xóm là `(r + dr, c + dc)`. Ô ở mép hoặc góc lưới có ít hàng xóm hơn, nên luôn phải kiểm tra **còn trong lưới không**: `0 <= nr < m and 0 <= nc < n`.

Một tập ô đen là **liên thông** (connected) nếu từ một ô đen bất kỳ có thể đi tới mọi ô đen khác, mỗi bước sang một hàng xóm **đen**. Một nhóm ô đen liên thông "lớn nhất có thể" gọi là một **thành phần liên thông** (connected component) — hay gọi vui là một "hòn đảo". Đề bảo đảm ảnh chỉ có **đúng một** thành phần đen.

Cell dưới in hàng xóm của vài ô, rồi vẽ 3 ảnh nhỏ: **(A)** liên thông; **(B)** hai ô chỉ chạm **góc** → **không** liên thông (2 thành phần); **(C)** hai khối cách nhau bởi các cột trắng → 2 thành phần. Ảnh (B), (C) **không thể** là input của bài này — nhưng chúng sẽ giúp ta hiểu vì sao điều kiện "liên thông" lại quan trọng đến vậy.

In [5]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải


def neighbors(r, c, m, n):
    """Các hàng xóm 4 hướng của (r, c) mà còn nằm trong lưới m x n."""
    result = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            result.append((nr, nc))
    return result


print("Lưới 3 x 4:")
print("  hàng xóm của (1, 1) (ở giữa)  :", neighbors(1, 1, 3, 4))
print("  hàng xóm của (0, 0) (góc)     :", neighbors(0, 0, 3, 4))
print("  hàng xóm của (2, 3) (góc)     :", neighbors(2, 3, 3, 4))
print("  hàng xóm của (0, 2) (mép trên):", neighbors(0, 2, 3, 4))
print()
show_grid(parse_image(["##.", ".#.", ".##"]), title="(A) liên thông: (0,0)->(0,1)->(1,1)->(2,1)->(2,2)")
show_grid(parse_image(["#..", ".#.", "..."]), title="(B) chỉ chạm góc: (0,0) và (1,1) KHÔNG kề -> 2 thành phần")
show_grid(parse_image(["##..#", "##..#"]), title="(C) hai khối cách nhau bởi cột trắng -> 2 thành phần")

Lưới 3 x 4:
  hàng xóm của (1, 1) (ở giữa)  : [(0, 1), (2, 1), (1, 0), (1, 2)]
  hàng xóm của (0, 0) (góc)     : [(1, 0), (0, 1)]
  hàng xóm của (2, 3) (góc)     : [(1, 3), (2, 2)]
  hàng xóm của (0, 2) (mép trên): [(1, 2), (0, 1), (0, 3)]

(A) liên thông: (0,0)->(0,1)->(1,1)->(2,1)->(2,2)
     0 1 2
  0  # # .
  1  . # .
  2  . # #
(B) chỉ chạm góc: (0,0) và (1,1) KHÔNG kề -> 2 thành phần
     0 1 2
  0  # . .
  1  . # .
  2  . . .
(C) hai khối cách nhau bởi cột trắng -> 2 thành phần
     0 1 2 3 4
  0  # # . . #
  1  # # . . #


## 5. Cách 1 — Quét toàn bộ ảnh (cách ai cũng nghĩ ra)

Ý tưởng đi thẳng từ định nghĩa ở mục 3: đi qua **mọi** ô `(r, c)`; gặp ô đen thì cập nhật 4 biên:

```text
top  = min(top, r)        bottom = max(bottom, r)
left = min(left, c)       right  = max(right, c)
```

Giá trị khởi đầu: `top` và `left` phải **lớn hơn mọi chỉ số hợp lệ** (để lần `min` đầu tiên chắc chắn lấy giá trị của ô đen đầu tiên), còn `bottom` và `right` phải **nhỏ hơn mọi chỉ số**. Ta dùng `top = m`, `left = n`, `bottom = right = -1`. (Cũng có thể dùng `float('inf')` — "vô cực" trong Python.)

Cách này **không cần** dùng `(x, y)`, cũng **không cần** tính liên thông — nó đúng với **mọi** ảnh có ít nhất một ô đen. Chính vì đơn giản và chắc chắn đúng nên ta dùng nó làm **trọng tài** (reference) để kiểm tra các cách nhanh hơn ở sau. Hàm vẫn nhận `x, y` để có cùng "chữ ký" với các cách khác.

In [6]:
def min_area_scan(image, x, y):
    m, n = len(image), len(image[0])
    top, bottom, left, right = m, -1, n, -1
    for r in range(m):
        for c in range(n):
            if image[r][c] == '1':
                top = min(top, r)
                bottom = max(bottom, r)
                left = min(left, c)
                right = max(right, c)
    return (bottom - top + 1) * (right - left + 1)


print("Example 1:", min_area_scan(image1, x1, y1), "(mong đợi 6)")
print("Example 2:", min_area_scan(image2, x2, y2), "(mong đợi 1)")
print("image_big:", min_area_scan(image_big, x_big, y_big))

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)
image_big: 28


### Trace cách 1 trên Example 1

Quét theo thứ tự: hàng 0 (trái → phải), rồi hàng 1, rồi hàng 2. Bảng chỉ ghi lúc gặp ô đen (gặp ô trắng thì không làm gì):

| ô đen gặp | top | bottom | left | right | ghi chú |
|---|---:|---:|---:|---:|---|
| (khởi đầu) | 3 | -1 | 4 | -1 | `top = m = 3`, `left = n = 4` |
| `(0,2)` | 0 | 0 | 2 | 2 | ô đen đầu tiên: cả 4 biên đều đổi |
| `(1,1)` | 0 | 1 | 1 | 2 | `bottom` xuống 1, `left` sang 1 |
| `(1,2)` | 0 | 1 | 1 | 2 | không đổi gì |
| `(2,1)` | 0 | 2 | 1 | 2 | `bottom` xuống 2 |

Kết quả `(2 - 0 + 1) * (2 - 1 + 1) = 6`. Dù chỉ có 4 ô đen, ta vẫn phải **đọc đủ 12 ô**.

In [7]:
def trace_scan(image):
    m, n = len(image), len(image[0])
    top, bottom, left, right = m, -1, n, -1
    reads = 0
    print(f"khởi đầu: top={top} bottom={bottom} left={left} right={right}")
    for r in range(m):
        for c in range(n):
            reads += 1
            if image[r][c] == '1':
                top, bottom = min(top, r), max(bottom, r)
                left, right = min(left, c), max(right, c)
                print(f"gặp ô đen ({r},{c}) -> top={top} bottom={bottom} left={left} right={right}")
    area = (bottom - top + 1) * (right - left + 1)
    print(f"đã đọc {reads} ô; diện tích = ({bottom}-{top}+1) * ({right}-{left}+1) = {area}")


trace_scan(image1)

khởi đầu: top=3 bottom=-1 left=4 right=-1
gặp ô đen (0,2) -> top=0 bottom=0 left=2 right=2
gặp ô đen (1,1) -> top=0 bottom=1 left=1 right=2
gặp ô đen (1,2) -> top=0 bottom=1 left=1 right=2
gặp ô đen (2,1) -> top=0 bottom=2 left=1 right=2
đã đọc 12 ô; diện tích = (2-0+1) * (2-1+1) = 6


### Độ phức tạp và cách 1 vướng ở đâu?

- **Thời gian `O(mn)`**: hai vòng lặp lồng nhau đọc đủ `m × n` ô. Với `m = n = 100` là 10 000 lần đọc.
- **Bộ nhớ `O(1)`**: chỉ 4 biến.

Vướng ở đâu?

1. Đề **cấm** đúng độ phức tạp này: "less than `O(mn)`".
2. Phần lớn công sức là đọc **ô trắng** vô ích. Ảnh 100 × 100 mà vùng đen chỉ có 5 ô thì 9 995 lần đọc là phí.
3. Ta **chưa dùng** hai "món quà" đề cho: tọa độ `(x, y)` của một ô đen, và việc vùng đen **liên thông**. Khi đề cho thêm thông tin mà lời giải không dùng tới, đó thường là dấu hiệu còn cách tốt hơn.

Câu hỏi dẫn đường: *"Đã biết sẵn một ô đen, có cách nào chỉ đi trên các ô đen, không đụng tới phần trắng mênh mông không?"* → Cách 2.

## 6. Nền tảng: BFS (tìm kiếm theo chiều rộng), `deque` và `visited`

### 6.1. Ý tưởng "vết dầu loang" (flood fill)

Nhỏ một giọt mực vào ô `(x, y)`: mực lan sang các hàng xóm **đen**, rồi từ đó lan tiếp sang hàng xóm đen của chúng... cho tới khi hết chỗ lan. Vì vùng đen liên thông, mực sẽ phủ **đúng toàn bộ** vùng đen. Ô trắng chỉ bị "nhìn" (để biết không lan qua được) chứ không bị tô. Thuật toán mô phỏng việc lan này gọi là **BFS** (Breadth-First Search — tìm kiếm theo chiều rộng): mực lan theo từng "vòng", ô gần được tô trước, ô xa được tô sau.

### 6.2. Hàng đợi (queue) và `collections.deque`

BFS cần một **hàng đợi** — giống xếp hàng mua vé: ai đến trước được phục vụ trước (**FIFO** — First In, First Out). Ô mới phát hiện được cho vào **cuối** hàng; ô được lấy ra xử lý từ **đầu** hàng.

Python có sẵn `collections.deque` (đọc là "đéc", viết tắt của double-ended queue — hàng đợi hai đầu):

- `q = deque([a])` — tạo hàng đợi có sẵn phần tử `a`.
- `q.append(v)` — thêm `v` vào **cuối**, tốn `O(1)`.
- `q.popleft()` — lấy ra và xóa phần tử **đầu**, tốn `O(1)`.
- `while q:` — lặp chừng nào hàng đợi còn phần tử (hàng đợi rỗng được coi là `False`).

Vì sao không dùng list thường? `lst.pop(0)` (lấy phần tử đầu list) phải dời mọi phần tử còn lại sang trái một ô → tốn `O(độ dài)`, chậm khi hàng dài. `deque.popleft()` thì luôn `O(1)`.

### 6.3. Tập `visited` — đừng thăm một ô hai lần

Ô `(1,1)` là hàng xóm của `(1,2)`, mà `(1,2)` cũng là hàng xóm của `(1,1)`. Nếu không ghi nhớ, mực sẽ lan qua lại giữa hai ô này mãi mãi → **vòng lặp vô hạn**. Ta dùng một **set** (tập hợp) tên `visited` chứa các ô đã **đưa vào hàng đợi**:

- `visited.add((r, c))` — đánh dấu ô `(r, c)`.
- `(r, c) in visited` — kiểm tra đã đánh dấu chưa, trung bình `O(1)`.

`(r, c)` là một **tuple** (bộ giá trị không sửa được), nên dùng làm phần tử của set được. List như `[r, c]` thì **không** cho vào set được (vì list sửa được).

In [8]:
from collections import deque

q = deque(["An"])
q.append("Bình")
q.append("Chi")
print("hàng đợi:", list(q))
first = q.popleft()
print("phục vụ:", first, "| còn lại:", list(q))
q.append("Dũng")
print("Dũng vào cuối:", list(q), "| phục vụ tiếp:", q.popleft(), "| còn:", list(q))

visited = set()
visited.add((0, 2))
visited.add((1, 2))
visited.add((0, 2))           # thêm lần nữa: set không lưu trùng
print("visited =", visited)
print("(1, 2) in visited ->", (1, 2) in visited, "| (2, 2) in visited ->", (2, 2) in visited)

hàng đợi: ['An', 'Bình', 'Chi']
phục vụ: An | còn lại: ['Bình', 'Chi']
Dũng vào cuối: ['Bình', 'Chi', 'Dũng'] | phục vụ tiếp: Bình | còn: ['Chi', 'Dũng']
visited = {(0, 2), (1, 2)}
(1, 2) in visited -> True | (2, 2) in visited -> False


### 6.4. Khung BFS trên lưới (5 bước)

1. Cho ô xuất phát vào hàng đợi **và** đánh dấu `visited` ngay.
2. Khi hàng đợi còn phần tử: `popleft()` lấy ô `(r, c)` ra **xử lý** (ở bài này: cập nhật 4 biên).
3. Xét 4 hàng xóm `(nr, nc)` của nó.
4. Hàng xóm nào **trong lưới**, **đen**, **chưa visited** → đánh dấu `visited` rồi `append` vào hàng đợi.
5. Hàng đợi rỗng → đã đi hết vùng.

Điểm tinh tế: đánh dấu `visited` **lúc cho vào hàng đợi** (bước 1 và 4), không phải lúc lấy ra. Như vậy mỗi ô vào hàng đợi **đúng một lần** (mục 17 sẽ cho thấy đánh dấu muộn thì chuyện gì xảy ra).

Cell dưới chạy BFS trên `image_big` từ `(2, 5)`, và thay vì tô đen, in ra ở mỗi ô đen **khoảng cách** (số bước đi ít nhất) từ ô xuất phát. Ta dùng một dict `dist` vừa làm `visited` (ô có trong dict = đã thăm) vừa lưu khoảng cách: ô mới phát hiện từ ô `(r, c)` có khoảng cách `dist[(r, c)] + 1`. Bạn sẽ thấy mực lan thành từng "vòng" 0, 1, 2, ...

In [9]:
def bfs_levels(image, x, y):
    """BFS từ (x, y); trả dict {(r, c): số bước từ (x, y)} cho mọi ô đen tới được."""
    m, n = len(image), len(image[0])
    dist = {(x, y): 0}            # dict này vừa là visited vừa lưu khoảng cách
    q = deque([(x, y)])
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(r, c, m, n):
            if image[nr][nc] == '1' and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return dist


dist = bfs_levels(image_big, x_big, y_big)
print("Khoảng cách BFS từ (2, 5) tới từng ô đen (ô trắng in '.'):")
print("     " + " ".join(str(c) for c in range(len(image_big[0]))))
for r in range(len(image_big)):
    line = " ".join(str(dist[(r, c)]) if (r, c) in dist else "." for c in range(len(image_big[0])))
    print(f"{r:>3}  " + line)
print("số ô đen BFS đi tới:", len(dist), "| tổng số ô đen trong ảnh:", sum(row.count('1') for row in image_big))

Khoảng cách BFS từ (2, 5) tới từng ô đen (ô trắng in '.'):
     0 1 2 3 4 5 6 7 8 9
  0  . . . . . . . . . .
  1  . . . 3 2 1 . . . .
  2  . . . 4 . 0 1 2 . .
  3  . 7 6 5 . . . 3 . .
  4  . . . . . . . 4 . .
  5  . . . . . . . . . .
số ô đen BFS đi tới: 12 | tổng số ô đen trong ảnh: 12


## 7. Cách 2 — BFS từ `(x, y)`, chỉ đi trên ô đen

Ghép khung BFS với việc cập nhật 4 biên: mỗi ô đen được lấy ra khỏi hàng đợi **đúng một lần**, và lúc đó ta cập nhật `top / bottom / left / right` y như cách 1. Khác biệt: cách 1 đọc **mọi** ô của ảnh; cách 2 chỉ đọc các ô đen và các ô trắng **sát viền** vùng đen.

Vì sao chắc chắn không sót ô đen nào? Vì vùng đen liên thông: mọi ô đen đều có đường đi (qua các ô đen) từ `(x, y)`, và BFS lần theo mọi đường như vậy. (Nếu ảnh có nhiều vùng, BFS chỉ thấy vùng chứa `(x, y)` — bài tập 6 sẽ khai thác điều này.)

Vài chi tiết code:

- Khởi đầu 4 biên bằng chính `(x, y)`: `top = bottom = x`, `left = right = y`, vì ta biết chắc đó là ô đen. Cú pháp `top = bottom = x` gán cùng một giá trị cho hai biến.
- **Thứ tự điều kiện quan trọng**: `0 <= nr < m and 0 <= nc < n and image[nr][nc] == '1'`. Toán tử `and` dừng ngay khi gặp vế `False` đầu tiên (short-circuit), nên ô ngoài lưới bị loại **trước khi** ta đọc `image[nr][nc]`. Nếu đảo thứ tự: chỉ số quá lớn gây `IndexError`, còn chỉ số **âm** như `-1` thì Python **không báo lỗi** mà lặng lẽ đọc ô cuối hàng (`row[-1]`) → sai âm thầm, rất khó tìm!

In [10]:
def min_area_bfs(image, x, y):
    m, n = len(image), len(image[0])
    top = bottom = x
    left = right = y
    visited = {(x, y)}
    q = deque([(x, y)])
    while q:
        r, c = q.popleft()
        top, bottom = min(top, r), max(bottom, r)
        left, right = min(left, c), max(right, c)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == '1' and (nr, nc) not in visited:
                visited.add((nr, nc))
                q.append((nr, nc))
    return (bottom - top + 1) * (right - left + 1)


print("Example 1:", min_area_bfs(image1, x1, y1), "(mong đợi 6)")
print("Example 2:", min_area_bfs(image2, x2, y2), "(mong đợi 1)")
print("image_big:", min_area_bfs(image_big, x_big, y_big), "| cách 1 cho:", min_area_scan(image_big, x_big, y_big))

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)
image_big: 28 | cách 1 cho: 28


### Trace cách 2 trên Example 1, xuất phát `(0, 2)`

Thứ tự xét hàng xóm: lên, xuống, trái, phải (theo `DIRS`). Cột "biên" ghi `(top, bottom, left, right)` **sau khi** cập nhật bằng ô vừa lấy ra.

| bước | lấy ra | biên sau cập nhật | hàng xóm đen mới được thêm | hàng đợi sau bước |
|---:|---|---|---|---|
| 0 | — | (0, 0, 2, 2) | `(0,2)` là ô xuất phát | `[(0,2)]` |
| 1 | `(0,2)` | (0, 0, 2, 2) | `(1,2)` | `[(1,2)]` |
| 2 | `(1,2)` | (0, 1, 2, 2) | `(1,1)` (ô `(0,2)` đã visited) | `[(1,1)]` |
| 3 | `(1,1)` | (0, 1, 1, 2) | `(2,1)` | `[(2,1)]` |
| 4 | `(2,1)` | (0, 2, 1, 2) | — | `[]` |

Hàng đợi rỗng → diện tích `(2 - 0 + 1) * (2 - 1 + 1) = 6`. Chỉ 4 ô đen được xử lý. Nhưng để ý con số "đã đọc" ở cuối output: mỗi ô đen nhìn tới 4 hàng xóm, nên trên ảnh **nhỏ xíu** như Example 1, BFS đọc **14** lần — còn nhiều hơn 12 lần của cách 1! Lợi thế của BFS chỉ lộ ra khi ảnh lớn mà vùng đen nhỏ.

In [11]:
def trace_bfs(image, x, y):
    m, n = len(image), len(image[0])
    top = bottom = x
    left = right = y
    visited = {(x, y)}
    q = deque([(x, y)])
    reads, step = 0, 0
    print(f"bước 0: khởi đầu biên=({top},{bottom},{left},{right}) hàng đợi={list(q)}")
    while q:
        step += 1
        r, c = q.popleft()
        top, bottom = min(top, r), max(bottom, r)
        left, right = min(left, c), max(right, c)
        added = []
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n:
                reads += 1                               # đọc ô image[nr][nc]
                if image[nr][nc] == '1' and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    q.append((nr, nc))
                    added.append((nr, nc))
        print(f"bước {step}: lấy ({r},{c}) biên=({top},{bottom},{left},{right}) thêm={added} hàng đợi={list(q)}")
    area = (bottom - top + 1) * (right - left + 1)
    print(f"hàng đợi rỗng; đã đọc {reads} ô hàng xóm; diện tích = {area}")


trace_bfs(image1, x1, y1)

bước 0: khởi đầu biên=(0,0,2,2) hàng đợi=[(0, 2)]
bước 1: lấy (0,2) biên=(0,0,2,2) thêm=[(1, 2)] hàng đợi=[(1, 2)]
bước 2: lấy (1,2) biên=(0,1,2,2) thêm=[(1, 1)] hàng đợi=[(1, 1)]
bước 3: lấy (1,1) biên=(0,1,1,2) thêm=[(2, 1)] hàng đợi=[(2, 1)]
bước 4: lấy (2,1) biên=(0,2,1,2) thêm=[] hàng đợi=[]
hàng đợi rỗng; đã đọc 14 ô hàng xóm; diện tích = 6


### 7.1. Biến thể DFS: đổi hàng đợi thành ngăn xếp (stack)

Nếu thay `popleft()` (lấy **đầu**) bằng `pop()` (lấy **cuối**), cấu trúc dữ liệu trở thành **stack** (ngăn xếp) — vào sau ra trước (**LIFO**, như chồng đĩa: đĩa đặt lên sau cùng được lấy ra trước). Khi đó ta đi "sâu" theo một hướng trước rồi mới quay lại: đó là **DFS** (Depth-First Search — tìm kiếm theo chiều sâu). Với bài này thứ tự thăm **không quan trọng** — chỉ cần thăm đủ mọi ô đen — nên DFS cho cùng đáp án. List thường làm stack là đủ: `append` và `pop()` ở cuối list đều `O(1)`.

DFS cũng hay được viết bằng **đệ quy** (recursion — hàm tự gọi lại chính nó) rất ngắn gọn. Nhưng cẩn thận: Python giới hạn độ sâu đệ quy (mặc định khoảng 1000 tầng). Ảnh `100 × 100` toàn đen có thể khiến đệ quy sâu hàng nghìn tầng → lỗi `RecursionError` (mục 17 sẽ chạy thử). Vì vậy ta viết DFS bằng **stack tự quản lý**, không đệ quy.

In [12]:
def min_area_dfs(image, x, y):
    m, n = len(image), len(image[0])
    top = bottom = x
    left = right = y
    visited = {(x, y)}
    stack = [(x, y)]
    while stack:
        r, c = stack.pop()               # khác BFS duy nhất ở dòng này: lấy phần tử CUỐI
        top, bottom = min(top, r), max(bottom, r)
        left, right = min(left, c), max(right, c)
        for nr, nc in neighbors(r, c, m, n):
            if image[nr][nc] == '1' and (nr, nc) not in visited:
                visited.add((nr, nc))
                stack.append((nr, nc))
    return (bottom - top + 1) * (right - left + 1)


for name, img, xx, yy in [("Example 1", image1, x1, y1), ("Example 2", image2, x2, y2),
                          ("image_big", image_big, x_big, y_big)]:
    print(f"{name}: dfs = {min_area_dfs(img, xx, yy)}, bfs = {min_area_bfs(img, xx, yy)}, "
          f"scan = {min_area_scan(img, xx, yy)}")

Example 1: dfs = 6, bfs = 6, scan = 6
Example 2: dfs = 1, bfs = 1, scan = 1
image_big: dfs = 28, bfs = 28, scan = 28


### Độ phức tạp cách 2 và nó vướng ở đâu?

Gọi `B` là số pixel đen (`1 <= B <= mn`).

- **Thời gian `O(B)`**: mỗi ô đen vào và ra hàng đợi đúng một lần, mỗi lần nhìn tối đa 4 hàng xóm → khoảng `4B` lần đọc.
- **Bộ nhớ `O(B)`**: tập `visited` và hàng đợi.

So với cách 1: rất tốt khi vùng đen **nhỏ** (ảnh 100 × 100 mà `B = 5` → chỉ khoảng 20 lần đọc thay vì 10 000).

Nhưng **trường hợp xấu nhất** vẫn tệ: nếu ảnh gần như toàn đen thì `B ≈ mn` → vẫn là `O(mn)` thời gian, lại còn tốn thêm `O(mn)` bộ nhớ. Đề đòi "nhanh hơn `O(mn)`" cho **mọi** ảnh, nên cách 2 chưa đạt.

Câu hỏi dẫn đường: *"Ta có thật sự cần biết **từng** ô đen không? Hay chỉ cần biết 4 con số `top, bottom, left, right`?"* → Chỉ cần 4 con số! Mục tiếp theo tìm cách xác định chúng mà **không** đi hết vùng đen.

## 8. Bước đột phá: nhìn "hình chiếu" (projection) của vùng đen

Nhớ lại mục 3: `left / right` chỉ phụ thuộc vào **cột nào có ô đen**, không cần biết ô đen nằm ở hàng nào. Vậy hãy "ép dẹt" ảnh theo chiều dọc: với mỗi cột `c`, chỉ ghi một bit — **cột `c` có ít nhất một ô đen không?** Giống như rọi đèn từ trên xuống rồi nhìn **cái bóng** của vùng đen in trên sàn. Dãy bit đó gọi là **hình chiếu lên trục cột** (column projection). Tương tự, ép dẹt theo chiều ngang ta được **hình chiếu lên trục hàng**.

Khi đó:

- `left` = vị trí **đầu tiên** có bit 1 trong hình chiếu cột; `right` = vị trí **cuối cùng** có bit 1.
- `top`, `bottom` = vị trí đầu tiên / cuối cùng có bit 1 trong hình chiếu hàng.

Bài toán 2D tách thành **hai bài 1D**: tìm đầu và cuối của các bit 1 trong một dãy. Cell dưới in ảnh kèm hình chiếu: hàng dưới cùng là hình chiếu cột, cột bên phải là hình chiếu hàng. (Hàm `col_projection` ở đây đọc **cả ảnh** — chỉ để minh họa; lời giải thật sẽ không tạo cả dãy này.)

In [13]:
def col_projection(image):
    """Dãy 0/1: cột c có ít nhất một ô đen không? (đọc cả ảnh - chỉ dùng để minh họa)"""
    m, n = len(image), len(image[0])
    proj = []
    for c in range(n):
        has = 0
        for r in range(m):
            if image[r][c] == '1':
                has = 1
        proj.append(has)
    return proj


def row_projection(image):
    return [1 if '1' in row else 0 for row in image]


def show_projection(image, title=None):
    m, n = len(image), len(image[0])
    cp, rp = col_projection(image), row_projection(image)
    if title:
        print(title)
    print("     " + " ".join(str(c % 10) for c in range(n)) + "   | hàng có đen?")
    for r in range(m):
        print(f"{r:>3}  " + " ".join("#" if v == '1' else "." for v in image[r]) + f"   | {rp[r]}")
    print("     " + "-" * (2 * n - 1))
    print("     " + " ".join(map(str, cp)) + "   <- cột có đen?")


show_projection(image1, "Example 1:")
print()
show_projection(image_big, "image_big:")

Example 1:
     0 1 2 3   | hàng có đen?
  0  . . # .   | 1
  1  . # # .   | 1
  2  . # . .   | 1
     -------
     0 1 1 0   <- cột có đen?

image_big:
     0 1 2 3 4 5 6 7 8 9   | hàng có đen?
  0  . . . . . . . . . .   | 0
  1  . . . # # # . . . .   | 1
  2  . . . # . # # # . .   | 1
  3  . # # # . . . # . .   | 1
  4  . . . . . . . # . .   | 1
  5  . . . . . . . . . .   | 0
     -------------------
     0 1 1 1 1 1 1 1 0 0   <- cột có đen?


### 8.1. Quan sát vàng: hình chiếu là MỘT đoạn liền, và đoạn đó chứa `y`

Nhìn `image_big`: hình chiếu cột là `0 1 1 1 1 1 1 1 0 0` — các bit 1 **dồn thành một khối liền**, không có lỗ hổng kiểu `1 1 0 1`. Đây không phải may mắn mà là **hệ quả bắt buộc của tính liên thông**:

> **Khẳng định.** Nếu vùng đen liên thông 4 hướng, gọi `L` = cột trái nhất có ô đen, `R` = cột phải nhất có ô đen, thì **mọi** cột `c` với `L <= c <= R` đều có ô đen.

**Lý do (bằng lời).** Lấy một ô đen `P` nằm ở cột `L` và một ô đen `Q` nằm ở cột `R`. Vì vùng đen liên thông, có một đường đi từ `P` tới `Q` chỉ qua các ô đen. Mỗi bước đi lên/xuống thì **giữ nguyên** cột; đi trái/phải thì cột thay đổi **đúng 1**. Vậy khi đi từ cột `L` sang cột `R`, chỉ số cột thay đổi từng nấc một — giống leo cầu thang mà không được nhảy bậc — nên nó phải **dừng chân ở mọi cột** `L, L+1, ..., R`. Ô dừng chân là ô đen ⇒ cột đó có ô đen. ∎

Lập luận y hệt cho hàng. Và vì `(x, y)` là một ô đen nên `L <= y <= R` và `top <= x <= bottom`: điểm đề cho **luôn nằm trong** đoạn liền.

Cell dưới kiểm chứng trên `image_big`: tìm một đường đi đen (bằng BFS có ghi lại "ô cha" — ô mà từ đó ta phát hiện ra ô hiện tại) từ ô `(3, 1)` ở cột trái nhất tới ô `(4, 7)` ở cột phải nhất, rồi in dãy cột trên đường đi: mọi cột 1..7 đều xuất hiện.

### 8.2. Phản ví dụ: khi KHÔNG liên thông thì hình chiếu có "lỗ"

Cell dưới cũng vẽ ảnh `image_split` giống ảnh (C) ở mục 4: hai khối cách nhau bởi các cột trắng. Hình chiếu cột thành `1 1 0 0 0 1` — **có lỗ hổng** ở giữa. Khi đó "các cột có đen" không còn là một đoạn liền, và (như ta sẽ thấy ở bài tập 2) cách binary search sẽ **trả sai**. Đây là lý do đề phải nói rõ *"the black pixels only form one component"*: điều kiện đó không phải trang trí, nó là **nền móng** của lời giải tối ưu.

In [14]:
def bfs_path(image, start, goal):
    """Một đường đi ngắn nhất qua các ô đen từ start tới goal (BFS + ghi ô cha)."""
    m, n = len(image), len(image[0])
    parent = {start: None}
    q = deque([start])
    while q:
        cur = q.popleft()
        if cur == goal:
            break
        for nxt in neighbors(cur[0], cur[1], m, n):
            if image[nxt[0]][nxt[1]] == '1' and nxt not in parent:
                parent[nxt] = cur
                q.append(nxt)
    path = []
    cur = goal
    while cur is not None:            # lần ngược từ goal về start theo ô cha
        path.append(cur)
        cur = parent[cur]
    return path[::-1]                 # [::-1] đảo ngược list: start ... goal


path = bfs_path(image_big, (3, 1), (4, 7))
print("đường đi    :", path)
print("dãy cột     :", [c for r, c in path])
print("các cột ghé :", sorted(set(c for r, c in path)), "-> đủ mọi cột từ 1 tới 7")
print()
image_split = parse_image([
    "##....",
    "##...#",
    ".....#",
])
show_projection(image_split, "image_split (KHÔNG liên thông - không phải input hợp lệ của đề):")
print("hình chiếu cột:", col_projection(image_split), "-> có lỗ ở cột 2, 3, 4")

đường đi    : [(3, 1), (3, 2), (3, 3), (2, 3), (1, 3), (1, 4), (1, 5), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
dãy cột     : [1, 2, 3, 3, 3, 4, 5, 5, 6, 7, 7, 7]
các cột ghé : [1, 2, 3, 4, 5, 6, 7] -> đủ mọi cột từ 1 tới 7

image_split (KHÔNG liên thông - không phải input hợp lệ của đề):
     0 1 2 3 4 5   | hàng có đen?
  0  # # . . . .   | 1
  1  # # . . . #   | 1
  2  . . . . . #   | 1
     -----------
     1 1 0 0 0 1   <- cột có đen?
hình chiếu cột: [1, 1, 0, 0, 0, 1] -> có lỗ ở cột 2, 3, 4


### 8.3. Từ "đoạn liền" tới "tính đơn điệu" (monotonic)

Cắt hình chiếu cột tại `y` thành hai nửa. Với `image_big`, `y = 5`:

```text
cột c        : 0 1 2 3 4 | 5 | 6 7 8 9
cột có đen?  : 0 1 1 1 1 | 1 | 1 1 0 0
               nửa trái    y   nửa phải
```

- **Nửa trái** `[0, y]`: dạng `0 0 ... 0 1 1 ... 1` — trắng hết rồi đen hết (vì đoạn đen liền và kéo dài tới tận `y`). Biên `left` = vị trí **chuyển từ 0 sang 1**.
- **Nửa phải** `[y, n-1]`: dạng `1 1 ... 1 0 0 ... 0` — đen hết rồi trắng hết. Biên `right` = ô **cuối cùng** còn 1, tức là ngay trước chỗ chuyển sang 0.

Một dãy chỉ đổi giá trị **đúng một lần** (kiểu `F F F T T T`) gọi là **đơn điệu** (monotonic). Tìm "chỗ đổi" trong một dãy đơn điệu là sở trường của **binary search**: không cần nhìn hết dãy, chỉ cần nhìn khoảng `log₂(độ dài)` vị trí. Mỗi lần "nhìn" một vị trí ở đây là hỏi *"cột `c` có ô đen không?"* — tốn `O(m)` (duyệt một cột) — nhưng ta chỉ hỏi khoảng `log₂ n` lần thay vì `n` lần.

Nếu ảnh không liên thông (như `image_split` với `y = 5`) thì nửa trái là `1 1 0 0 0 | 1` — đổi giá trị **hai lần**, không đơn điệu → binary search mất phương hướng.

## 9. Nền tảng: binary search (tìm kiếm nhị phân) từ căn bản

### 9.1. Trò chơi đoán số

Bạn nghĩ một số từ 1 tới 100, tôi đoán; mỗi lần bạn chỉ trả lời "lớn hơn" hoặc "nhỏ hơn". Đoán lần lượt 1, 2, 3, ... thì có thể mất 100 lần. Đoán **chính giữa** (50) thì mỗi câu trả lời loại được **một nửa** số khả năng: 100 → 50 → 25 → 13 → 7 → 4 → 2 → 1, chỉ khoảng **7 lần**. Đó là binary search: mỗi bước chia đôi vùng tìm kiếm, nên số bước chỉ khoảng `log₂(kích thước)`.

### 9.2. Mẫu quan trọng nhất: tìm vị trí `True` đầu tiên

Cho dãy `ok` **đơn điệu** dạng `F F ... F T T ... T`. Tìm chỉ số của `True` đầu tiên (nếu không có `True` nào thì trả `len(ok)`).

```python
lo, hi = 0, len(ok)          # đáp án nằm trong [lo, hi]
while lo < hi:               # còn hơn 1 khả năng
    mid = (lo + hi) // 2     # điểm giữa, làm tròn xuống
    if ok[mid]:              # mid là True -> đáp án là mid hoặc ở bên trái mid
        hi = mid             #   KHÔNG được bỏ mid: có thể chính nó là True đầu tiên
    else:                    # mid là False -> mọi vị trí <= mid đều False (vì đơn điệu)
        lo = mid + 1         #   -> bỏ luôn mid
return lo                    # lo == hi: chỉ còn 1 khả năng, đó là đáp án
```

**Bất biến (invariant)** — điều luôn đúng ở đầu mỗi vòng lặp:

- mọi vị trí `< lo` đều là `False`;
- vị trí `hi` là `True` — hoặc `hi = len(ok)`, coi như một "ô `True` ảo" nằm ngay ngoài dãy. Ô ảo này gọi là **lính canh** (sentinel).

Khi vòng lặp dừng (`lo == hi`), hai điều trên nói rằng `lo` chính là `True` đầu tiên (hoặc là lính canh nếu không có `True` nào).

Ba chi tiết hay sai:

1. `hi = mid` (không phải `mid - 1`), vì `mid` có thể chính là đáp án.
2. `lo = mid + 1` (không phải `mid`). Nếu viết `lo = mid`, khi `hi = lo + 1` thì `mid = lo` và vùng tìm kiếm **đứng yên mãi** → lặp vô hạn.
3. Vì `mid = (lo + hi) // 2` làm tròn xuống, khi `lo < hi` thì luôn có `lo <= mid < hi` → ta **không bao giờ đọc** vị trí `hi`. Nhờ vậy `hi` được phép là "ô ảo" `len(ok)` mà không bị lỗi chỉ số. Chi tiết nhỏ này sẽ "cứu" ta khi tìm biên phải ở mục 11.

In [15]:
def first_true_list(ok, verbose=False):
    lo, hi = 0, len(ok)
    while lo < hi:
        mid = (lo + hi) // 2
        if verbose:
            print(f"  lo={lo} hi={hi} mid={mid} ok[mid]={ok[mid]!s:<5}",
                  "-> hi = mid" if ok[mid] else "-> lo = mid + 1")
        if ok[mid]:
            hi = mid
        else:
            lo = mid + 1
    return lo


F, T = False, True
for ok in ([F, F, F, F, F, T, T, T], [T, T, T, T], [F, F, F]):
    print("ok =", " ".join("T" if v else "F" for v in ok))
    print("  => True đầu tiên ở chỉ số", first_true_list(ok, verbose=True))

ok = F F F F F T T T
  lo=0 hi=8 mid=4 ok[mid]=False -> lo = mid + 1
  lo=5 hi=8 mid=6 ok[mid]=True  -> hi = mid
  lo=5 hi=6 mid=5 ok[mid]=True  -> hi = mid
  => True đầu tiên ở chỉ số 5
ok = T T T T
  lo=0 hi=4 mid=2 ok[mid]=True  -> hi = mid
  lo=0 hi=2 mid=1 ok[mid]=True  -> hi = mid
  lo=0 hi=1 mid=0 ok[mid]=True  -> hi = mid
  => True đầu tiên ở chỉ số 0
ok = F F F
  lo=0 hi=3 mid=1 ok[mid]=False -> lo = mid + 1
  lo=2 hi=3 mid=2 ok[mid]=False -> lo = mid + 1
  => True đầu tiên ở chỉ số 3


### 9.3. Binary search nhanh cỡ nào?

Mỗi vòng, vùng `[lo, hi]` còn khoảng một nửa. Đi từ `N` khả năng xuống 1 khả năng cần khoảng `log₂ N` vòng. Cell dưới đo **số vòng nhiều nhất** (thử mọi vị trí có thể của đáp án) cho vài độ dài dãy: dãy dài gấp 10 lần thì chỉ tốn thêm khoảng 3–4 vòng.

Để ý mẹo nhỏ trong code: ta **không tạo dãy thật**, mà coi `ok[mid]` là câu hỏi `mid >= answer` và tính ngay tại chỗ. Binary search chỉ cần **trả lời được câu hỏi Có/Không tại vị trí `mid`** — không cần dãy có sẵn. Đây chính là điều ta sẽ làm với ảnh: câu hỏi là "cột `mid` có đen không?".

In [16]:
import math


def count_loops(N, answer):
    """Số vòng của mẫu 'True đầu tiên' trên dãy dài N có True đầu tiên ở vị trí `answer`."""
    lo, hi, loops = 0, N, 0
    while lo < hi:
        loops += 1
        mid = (lo + hi) // 2
        if mid >= answer:          # ok[mid] = (mid >= answer): tính tại chỗ, không cần dãy thật
            hi = mid
        else:
            lo = mid + 1
    return loops


for N in [10, 100, 1000, 10**6]:
    answers = range(N + 1) if N <= 1000 else [0, N // 3, N // 2, N]
    worst = max(count_loops(N, a) for a in answers)
    print(f"N = {N:>7}: nhiều nhất {worst:>2} vòng   (log2(N) ≈ {math.log2(N):.1f})")

N =      10: nhiều nhất  4 vòng   (log2(N) ≈ 3.3)
N =     100: nhiều nhất  7 vòng   (log2(N) ≈ 6.6)
N =    1000: nhiều nhất 10 vòng   (log2(N) ≈ 10.0)
N = 1000000: nhiều nhất 20 vòng   (log2(N) ≈ 19.9)


## 10. Hai "câu hỏi" binary search sẽ đặt ra: `has_black_in_col`, `has_black_in_row`

Ở mục 9, `ok[mid]` là một phần tử có sẵn. Ở bài này **không có** sẵn dãy hình chiếu (tạo cả dãy tốn `O(mn)` — đúng thứ ta muốn tránh). Thay vào đó, mỗi khi binary search cần biết "vị trí `mid` là 0 hay 1", ta **tính tại chỗ**:

- `has_black_in_row(image, r)`: hàng `r` có ô đen không? → `'1' in image[r]`, duyệt tối đa `n` ô → `O(n)`.
- `has_black_in_col(image, c)`: cột `c` có ô đen không? → xem lần lượt `image[0][c], image[1][c], ..., image[m-1][c]`, tối đa `m` ô → `O(m)`.

### Kiến thức Python: `any()` và generator expression

- `any(dãy)` trả `True` nếu **có ít nhất một** phần tử đúng; `any([])` (dãy rỗng) là `False`.
- `any(row[c] == '1' for row in image)`: phần trong ngoặc là một **generator expression** (biểu thức sinh) — trông giống list comprehension nhưng không có ngoặc vuông, không tạo cả list mà **sinh từng giá trị khi cần**. `for row in image` lấy lần lượt từng hàng, `row[c]` là ô cột `c` của hàng đó.
- `any` **dừng ngay** khi gặp giá trị `True` đầu tiên (short-circuit — "đoản mạch"). Vì vậy một cột có ô đen ở hàng 0 chỉ tốn 1 lần đọc; chỉ cột **trắng hoàn toàn** mới tốn đủ `m` lần.

Cell dưới in ra từng lần đọc để thấy `any` dừng sớm, rồi định nghĩa hai hàm câu hỏi.

In [17]:
print("any([False, True, False]) =", any([False, True, False]))
print("any([False, False])       =", any([False, False]))
print("any([])                   =", any([]))


def noisy_is_black(image, r, c):
    print(f"    đọc ô ({r},{c}) = {image[r][c]}")
    return image[r][c] == '1'


print("Cột 1 của Example 1 có đen?")
print("  ->", any(noisy_is_black(image1, r, 1) for r in range(len(image1))), "(dừng ngay ở hàng 1)")
print("Cột 3 của Example 1 có đen?")
print("  ->", any(noisy_is_black(image1, r, 3) for r in range(len(image1))), "(cột trắng: phải đọc đủ 3 ô)")


def has_black_in_row(image, r):
    return '1' in image[r]                         # O(n)


def has_black_in_col(image, c):
    return any(row[c] == '1' for row in image)     # O(m)


print("Example 1 - hàng có đen?", [has_black_in_row(image1, r) for r in range(3)])
print("Example 1 - cột có đen? ", [has_black_in_col(image1, c) for c in range(4)])
print("image_big - cột có đen? ", [int(has_black_in_col(image_big, c)) for c in range(10)])

any([False, True, False]) = True
any([False, False])       = False
any([])                   = False
Cột 1 của Example 1 có đen?
    đọc ô (0,1) = 0
    đọc ô (1,1) = 1
  -> True (dừng ngay ở hàng 1)
Cột 3 của Example 1 có đen?
    đọc ô (0,3) = 0
    đọc ô (1,3) = 0
    đọc ô (2,3) = 0
  -> False (cột trắng: phải đọc đủ 3 ô)
Example 1 - hàng có đen? [True, True, True]
Example 1 - cột có đen?  [False, True, True, False]
image_big - cột có đen?  [0, 1, 1, 1, 1, 1, 1, 1, 0, 0]


## 11. Cách 3 — Binary search 4 lần trên hình chiếu (cách chính)

Ghép mọi thứ lại. Cả 4 biên đều tìm bằng **đúng một mẫu** "vị trí `True` đầu tiên" của mục 9, chỉ khác vùng tìm và câu hỏi:

| biên | vùng tìm `[lo, hi]` | câu hỏi tại `mid` | dãy câu trả lời trông như | kết quả |
|---|---|---|---|---|
| `left` | `[0, y]` | cột `mid` có đen? | `F..F T..T` (cột `y` chắc chắn `T`) | `left = lo` |
| `right` | `[y+1, n]` | cột `mid` **trắng**? | `F..F T..T` (cột `n` là cột ảo, coi như trắng) | `right = lo - 1` |
| `top` | `[0, x]` | hàng `mid` có đen? | `F..F T..T` (hàng `x` chắc chắn `T`) | `top = lo` |
| `bottom` | `[x+1, m]` | hàng `mid` **trắng**? | `F..F T..T` (hàng `m` là hàng ảo) | `bottom = lo - 1` |

**Biên trái:** vùng `[0, y]`, và ta biết chắc cột `y` có đen → cột `y` đóng vai "vị trí `hi` là `True`" trong bất biến. Không cần hỏi cột `y`.

**Mẹo cho biên phải:** ở nửa phải, dãy "có đen?" là `T..T F..F` — **ngược chiều** mẫu. Thay vì học thêm mẫu "`True` cuối cùng" (có bẫy lặp vô hạn, xem lỗi 5 ở mục 17), ta **đổi câu hỏi** thành "cột này **trắng** không?" → dãy thành `F..F T..T`, đúng mẫu quen thuộc. Cột trắng đầu tiên bên phải `y` nằm **ngay sau** biên phải, nên `right = (cột trắng đầu tiên) - 1`. Nếu vùng đen chạm mép phải ảnh thì không có cột trắng nào → kết quả là `n` (cột ảo ngoài ảnh) → `right = n - 1`. Đúng như mục 9 đã nói, ta **không bao giờ đọc** cột `n` vì luôn có `mid < hi`.

**Hàng** làm y hệt, dùng `x` thay `y`, `m` thay `n`, `has_black_in_row` thay `has_black_in_col`.

Tại sao tìm trong `[0, y]` mà không phải cả `[0, n-1]`? Vì chỉ **từng nửa** mới đơn điệu! Cả dãy `0 1 1 1 1 1 1 1 0 0` thì đổi giá trị hai lần; binary search trên cả dãy có thể "trượt" vào phần `0` bên phải (mục 17, lỗi 4). Điểm `(x, y)` mà đề cho chính là **chỗ cắt** để mỗi nửa đơn điệu — món quà thứ hai của đề giờ mới được dùng tới.

In [18]:
def find_left(image, y):
    """Cột ĐEN đầu tiên trong [0, y]. Bất biến: mọi cột < lo trắng; cột hi đen."""
    lo, hi = 0, y
    while lo < hi:
        mid = (lo + hi) // 2
        if has_black_in_col(image, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def find_right(image, y):
    """Cột đen CUỐI CÙNG = (cột TRẮNG đầu tiên trong [y+1, n]) - 1."""
    n = len(image[0])
    lo, hi = y + 1, n                          # hi = n: cột ảo, coi như trắng, không bao giờ bị đọc
    while lo < hi:
        mid = (lo + hi) // 2
        if not has_black_in_col(image, mid):   # cột mid trắng?
            hi = mid
        else:
            lo = mid + 1
    return lo - 1


def find_top(image, x):
    """Hàng ĐEN đầu tiên trong [0, x]."""
    lo, hi = 0, x
    while lo < hi:
        mid = (lo + hi) // 2
        if has_black_in_row(image, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def find_bottom(image, x):
    """Hàng đen CUỐI CÙNG = (hàng TRẮNG đầu tiên trong [x+1, m]) - 1."""
    m = len(image)
    lo, hi = x + 1, m
    while lo < hi:
        mid = (lo + hi) // 2
        if not has_black_in_row(image, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo - 1


def min_area_binary(image, x, y):
    left, right = find_left(image, y), find_right(image, y)
    top, bottom = find_top(image, x), find_bottom(image, x)
    return (right - left + 1) * (bottom - top + 1)


print("Example 1:", min_area_binary(image1, x1, y1), "(mong đợi 6)")
print("Example 2:", min_area_binary(image2, x2, y2), "(mong đợi 1)")
print("image_big:", min_area_binary(image_big, x_big, y_big), "| cách 1 cho:", min_area_scan(image_big, x_big, y_big))
print("biên của image_big: left =", find_left(image_big, 5), " right =", find_right(image_big, 5),
      " top =", find_top(image_big, 2), " bottom =", find_bottom(image_big, 2))

Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)
image_big: 28 | cách 1 cho: 28
biên của image_big: left = 1  right = 7  top = 1  bottom = 4


### Trace cách 3 trên Example 1 (`x = 0, y = 2`)

```text
     0 1 2 3   | hàng có đen?
  0  . . @ .   | 1
  1  . # # .   | 1
  2  . # . .   | 1
     -------
     0 1 1 0   <- cột có đen?
```

| biên | `lo` | `hi` | `mid` | câu hỏi | trả lời | cập nhật |
|---|---:|---:|---:|---|---|---|
| left | 0 | 2 | 1 | cột 1 có đen? | `True` | `hi = 1` |
| left | 0 | 1 | 0 | cột 0 có đen? | `False` | `lo = 1` → dừng, `left = 1` |
| right | 3 | 4 | 3 | cột 3 trắng? | `True` | `hi = 3` → dừng, cột trắng đầu tiên = 3, `right = 2` |
| top | 0 | 0 | — | (vùng chỉ còn 1 khả năng) | — | không vòng nào, `top = 0` |
| bottom | 1 | 3 | 2 | hàng 2 trắng? | `False` | `lo = 3` → dừng, hàng trắng đầu tiên = 3 (hàng ảo), `bottom = 2` |

Diện tích `(2 - 1 + 1) * (2 - 0 + 1) = 2 * 3 = 6` ✔.

Hai chi tiết đáng để ý: (1) biên trên tìm trong `[0, 0]` — vùng chỉ có một khả năng nên vòng `while` không chạy lần nào; (2) biên dưới: vùng đen chạm mép dưới ảnh, nên "hàng trắng đầu tiên" là **hàng ảo** `m = 3` — và ta không hề đọc hàng 3 (vốn không tồn tại).

Hàm `trace_min_area_binary` dưới in đúng các bước trên. Để khỏi viết 4 lần vòng lặp có `print`, nó dùng **một** hàm `trace_search` chung, nhận thêm tham số `kind` cho biết đang hỏi câu nào (`"col_black"`, `"col_white"`, `"row_black"`, `"row_white"`). Hãy nhớ ý tưởng "một hàm chung + tham số" này — cách 4 sẽ làm nó gọn hơn nữa.

In [19]:
QUESTIONS = {
    "col_black": "cột {} có đen?",
    "col_white": "cột {} trắng?",
    "row_black": "hàng {} có đen?",
    "row_white": "hàng {} trắng?",
}


def ask(image, kind, i):
    if kind == "col_black":
        return has_black_in_col(image, i)
    if kind == "col_white":
        return not has_black_in_col(image, i)
    if kind == "row_black":
        return has_black_in_row(image, i)
    return not has_black_in_row(image, i)          # kind == "row_white"


def trace_search(image, lo, hi, kind):
    if lo == hi:
        print("   (lo == hi ngay từ đầu: không cần vòng nào)")
    while lo < hi:
        mid = (lo + hi) // 2
        ans = ask(image, kind, mid)
        action = "hi = mid" if ans else "lo = mid + 1"
        print(f"   lo={lo} hi={hi} mid={mid} | {QUESTIONS[kind].format(mid):<15} {ans!s:<5} -> {action}")
        if ans:
            hi = mid
        else:
            lo = mid + 1
    return lo


def trace_min_area_binary(image, x, y):
    m, n = len(image), len(image[0])
    print(f"[left]   cột ĐEN đầu tiên trong [0, {y}]  (cột {y} = y, chắc chắn đen)")
    left = trace_search(image, 0, y, "col_black")
    print(f"   => left = {left}")
    print(f"[right]  cột TRẮNG đầu tiên trong [{y + 1}, {n}]  (cột {n} = cột ảo, coi như trắng)")
    right_ex = trace_search(image, y + 1, n, "col_white")
    print(f"   => cột trắng đầu tiên = {right_ex}, right = {right_ex - 1}")
    print(f"[top]    hàng ĐEN đầu tiên trong [0, {x}]  (hàng {x} = x, chắc chắn đen)")
    top = trace_search(image, 0, x, "row_black")
    print(f"   => top = {top}")
    print(f"[bottom] hàng TRẮNG đầu tiên trong [{x + 1}, {m}]  (hàng {m} = hàng ảo, coi như trắng)")
    bottom_ex = trace_search(image, x + 1, m, "row_white")
    print(f"   => hàng trắng đầu tiên = {bottom_ex}, bottom = {bottom_ex - 1}")
    right, bottom = right_ex - 1, bottom_ex - 1
    area = (right - left + 1) * (bottom - top + 1)
    print(f"diện tích = ({right} - {left} + 1) * ({bottom} - {top} + 1) = {area}")


trace_min_area_binary(image1, x1, y1)

[left]   cột ĐEN đầu tiên trong [0, 2]  (cột 2 = y, chắc chắn đen)
   lo=0 hi=2 mid=1 | cột 1 có đen?   True  -> hi = mid
   lo=0 hi=1 mid=0 | cột 0 có đen?   False -> lo = mid + 1
   => left = 1
[right]  cột TRẮNG đầu tiên trong [3, 4]  (cột 4 = cột ảo, coi như trắng)
   lo=3 hi=4 mid=3 | cột 3 trắng?    True  -> hi = mid
   => cột trắng đầu tiên = 3, right = 2
[top]    hàng ĐEN đầu tiên trong [0, 0]  (hàng 0 = x, chắc chắn đen)
   (lo == hi ngay từ đầu: không cần vòng nào)
   => top = 0
[bottom] hàng TRẮNG đầu tiên trong [1, 3]  (hàng 3 = hàng ảo, coi như trắng)
   lo=1 hi=3 mid=2 | hàng 2 trắng?   False -> lo = mid + 1
   => hàng trắng đầu tiên = 3, bottom = 2
diện tích = (2 - 1 + 1) * (2 - 0 + 1) = 6


### Trace cách 3 trên `image_big` (`x = 2, y = 5`) — đủ cả hai nhánh

```text
     0 1 2 3 4 5 6 7 8 9   | hàng có đen?
  0  . . . . . . . . . .   | 0
  1  . . . # # # . . . .   | 1
  2  . . . # . @ # # . .   | 1
  3  . # # # . . . # . .   | 1
  4  . . . . . . . # . .   | 1
  5  . . . . . . . . . .   | 0
     -------------------
     0 1 1 1 1 1 1 1 0 0   <- cột có đen?
```

Ở ảnh này mỗi lần tìm đi qua nhiều vòng, có cả nhánh `hi = mid` lẫn `lo = mid + 1`:

| biên | các bước `(lo, hi, mid)` → trả lời → cập nhật | kết quả |
|---|---|---|
| left, vùng `[0, 5]` | `(0,5,2)` cột 2 có đen? `True` → `hi=2`; `(0,2,1)` cột 1 có đen? `True` → `hi=1`; `(0,1,0)` cột 0 có đen? `False` → `lo=1` | `left = 1` |
| right, vùng `[6, 10]` | `(6,10,8)` cột 8 trắng? `True` → `hi=8`; `(6,8,7)` cột 7 trắng? `False` → `lo=8` | cột trắng đầu = 8, `right = 7` |
| top, vùng `[0, 2]` | `(0,2,1)` hàng 1 có đen? `True` → `hi=1`; `(0,1,0)` hàng 0 có đen? `False` → `lo=1` | `top = 1` |
| bottom, vùng `[3, 6]` | `(3,6,4)` hàng 4 trắng? `False` → `lo=5`; `(5,6,5)` hàng 5 trắng? `True` → `hi=5` | hàng trắng đầu = 5, `bottom = 4` |

Diện tích `(7 - 1 + 1) * (4 - 1 + 1) = 7 * 4 = 28`. Tổng cộng chỉ hỏi **9** câu hỏi (mỗi câu đọc tối đa một hàng hoặc một cột) thay vì đọc cả 60 ô. Cell dưới in đúng các bước này.

In [20]:
trace_min_area_binary(image_big, x_big, y_big)

[left]   cột ĐEN đầu tiên trong [0, 5]  (cột 5 = y, chắc chắn đen)
   lo=0 hi=5 mid=2 | cột 2 có đen?   True  -> hi = mid
   lo=0 hi=2 mid=1 | cột 1 có đen?   True  -> hi = mid
   lo=0 hi=1 mid=0 | cột 0 có đen?   False -> lo = mid + 1
   => left = 1
[right]  cột TRẮNG đầu tiên trong [6, 10]  (cột 10 = cột ảo, coi như trắng)
   lo=6 hi=10 mid=8 | cột 8 trắng?    True  -> hi = mid
   lo=6 hi=8 mid=7 | cột 7 trắng?    False -> lo = mid + 1
   => cột trắng đầu tiên = 8, right = 7
[top]    hàng ĐEN đầu tiên trong [0, 2]  (hàng 2 = x, chắc chắn đen)
   lo=0 hi=2 mid=1 | hàng 1 có đen?  True  -> hi = mid
   lo=0 hi=1 mid=0 | hàng 0 có đen?  False -> lo = mid + 1
   => top = 1
[bottom] hàng TRẮNG đầu tiên trong [3, 6]  (hàng 6 = hàng ảo, coi như trắng)
   lo=3 hi=6 mid=4 | hàng 4 trắng?   False -> lo = mid + 1
   lo=5 hi=6 mid=5 | hàng 5 trắng?   True  -> hi = mid
   => hàng trắng đầu tiên = 5, bottom = 4
diện tích = (7 - 1 + 1) * (4 - 1 + 1) = 28


### Độ phức tạp cách 3

- Tìm `left` và `right`: mỗi lần khoảng `log₂ n` câu hỏi, mỗi câu `has_black_in_col` tốn `O(m)` → `O(m log n)`.
- Tìm `top` và `bottom`: mỗi lần khoảng `log₂ m` câu hỏi, mỗi câu `has_black_in_row` tốn `O(n)` → `O(n log m)`.
- **Tổng thời gian `O(m log n + n log m)`**, **bộ nhớ `O(1)`** (chỉ vài biến `lo, hi, mid`).

Ví dụ số: với `m = n = 100`, `log₂ 100 ≈ 7`, nên nhiều nhất khoảng `2·7·100 + 2·7·100 = 2800` lần đọc, so với `10 000` của cách 1. Nếu ảnh to `m = n = 10 000`: khoảng `4·14·10 000 = 560 000` lần đọc so với `10⁸` — nhanh hơn gần 200 lần. Và khác cách 2, con số này **không phụ thuộc** vùng đen to hay nhỏ.

Cell dưới **đếm số lần đọc ô thật sự** của 3 cách trên vài ảnh `100 × 100` (kích thước lớn nhất của đề). Để đếm, ta "bọc" mỗi hàng bằng class `CountingRow` — một list đặc biệt tự tăng bộ đếm mỗi khi bị đọc `row[c]` hoặc bị hỏi `'1' in row`. Bạn không cần hiểu chi tiết class này, chỉ cần đọc bảng kết quả.

In [21]:
class CountingRow(list):
    reads = 0                                    # bộ đếm dùng chung cho mọi hàng

    def __getitem__(self, c):                    # chạy mỗi khi viết row[c]
        CountingRow.reads += 1
        return list.__getitem__(self, c)

    def __contains__(self, value):               # chạy mỗi khi viết  value in row
        for c in range(len(self)):
            if self[c] == value:                 # self[c] đi qua __getitem__ nên cũng được đếm
                return True
        return False


def count_reads(func, image, x, y):
    wrapped = [CountingRow(row) for row in image]
    CountingRow.reads = 0
    result = func(wrapped, x, y)
    return result, CountingRow.reads


def make_image(m, n, cells):
    img = [['0'] * n for _ in range(m)]
    for r, c in cells:
        img[r][c] = '1'
    return img


shapes = [
    ("ô vuông 2x2 ở giữa", make_image(100, 100, [(49, 49), (49, 50), (50, 49), (50, 50)]), 50, 50),
    ("chữ thập", make_image(100, 100, [(50, c) for c in range(100)] + [(r, 50) for r in range(100)]), 50, 50),
    ("bậc thang chéo", make_image(100, 100, [(i, i) for i in range(100)] + [(i, i + 1) for i in range(99)]), 50, 50),
    ("toàn đen", [['1'] * 100 for _ in range(100)], 0, 0),
]
print(f"{'hình dạng (100x100)':<20}{'B':>6} | {'cách 1':>7} {'cách 2':>7} {'cách 3':>7}   <- số lần đọc ô")
for name, img, x, y in shapes:
    B = sum(row.count('1') for row in img)
    res = [count_reads(f, img, x, y) for f in (min_area_scan, min_area_bfs, min_area_binary)]
    assert res[0][0] == res[1][0] == res[2][0]          # 3 cách cùng đáp án
    print(f"{name:<20}{B:>6} | {res[0][1]:>7} {res[1][1]:>7} {res[2][1]:>7}")

hình dạng (100x100)      B |  cách 1  cách 2  cách 3   <- số lần đọc ô
ô vuông 2x2 ở giữa       4 |   10000      16    2300
chữ thập               199 |   10000     792    1122
bậc thang chéo         199 |   10000     790    1012
toàn đen             10000 |   10000   39600      12


## 12. Nền tảng: truyền hàm như một giá trị, `lambda`

Nhìn lại 4 hàm `find_left / find_right / find_top / find_bottom` ở mục 11: chúng **giống nhau tới 90%** — cùng vòng `while lo < hi`, cùng `mid`, cùng `hi = mid` / `lo = mid + 1`. Chỉ khác (1) vùng `[lo, hi]` và (2) câu hỏi đặt cho `mid`. Lặp code 4 lần thì dễ sửa chỗ này mà quên chỗ kia. Hàm `trace_search` đã gom lại bằng tham số `kind` (một chuỗi + chuỗi `if`). Python cho phép cách gọn hơn: **truyền thẳng câu hỏi (một hàm) vào làm tham số**.

Trong Python, hàm cũng là một **giá trị** như số hay chuỗi: gán được vào biến, truyền được vào hàm khác.

- `lambda v: v * v` là một **hàm vô danh** (anonymous function) viết trên một dòng: nhận `v`, trả `v * v`. Tương đương `def f(v): return v * v`. Cú pháp: `lambda tham_số: biểu_thức_trả_về`.
- `lambda c: has_black_in_col(image, c)` — hàm một tham số `c`, nhưng "nhớ" luôn biến `image` ở bên ngoài. Hàm nhớ biến của nơi nó được tạo ra như vậy gọi là **closure**.
- `lambda c: not has_black_in_col(image, c)` — câu hỏi ngược lại: "cột `c` trắng?".

Một hàm trả về `True/False` như vậy gọi là **predicate** (vị từ, hàm điều kiện).

In [22]:
square = lambda v: v * v
print("square(7) =", square(7))


def apply_all(f, values):
    """Gọi hàm f trên từng giá trị - f được truyền vào như một tham số bình thường."""
    return [f(v) for v in values]


print("bình phương:", apply_all(square, [1, 2, 3, 4]))
print("chẵn?      :", apply_all(lambda v: v % 2 == 0, [1, 2, 3, 4]))

col_black = lambda c: has_black_in_col(image1, c)        # closure: nhớ image1
col_white = lambda c: not has_black_in_col(image1, c)
print("Example 1 - cột có đen?:", apply_all(col_black, range(4)))
print("Example 1 - cột trắng? :", apply_all(col_white, range(4)))

square(7) = 49
bình phương: [1, 4, 9, 16]
chẵn?      : [False, True, False, True]
Example 1 - cột có đen?: [False, True, True, False]
Example 1 - cột trắng? : [True, False, False, True]


## 13. Cách 4 — Một hàm binary search tổng quát cho cả 4 biên

Ta viết **một** hàm duy nhất:

```python
def first_true(lo, hi, pred):
    """Vị trí nhỏ nhất i trong [lo, hi) có pred(i) True; không có thì trả hi.
       Yêu cầu: pred đơn điệu F..F T..T trên [lo, hi)."""
```

Cách đọc `[lo, hi)`: khoảng **nửa mở** — gồm `lo` nhưng **không gồm** `hi`. "Không có `True` nào thì trả `hi`" chính là ý "ô `hi` là lính canh `True`" ở mục 9 — hai cách nói của cùng một điều.

Khi đó 4 biên chỉ còn là 4 lời gọi:

| biên | lời gọi |
|---|---|
| `left` | `first_true(0, y, lambda c: has_black_in_col(image, c))` |
| `right_ex` (= `right + 1`) | `first_true(y + 1, n, lambda c: not has_black_in_col(image, c))` |
| `top` | `first_true(0, x, lambda r: has_black_in_row(image, r))` |
| `bottom_ex` (= `bottom + 1`) | `first_true(x + 1, m, lambda r: not has_black_in_row(image, r))` |

Hậu tố `_ex` nghĩa là **exclusive** — biên "không tính": `right_ex` là cột trắng đầu tiên **sau** đoạn đen, tức `right + 1`. Lợi ích: chiều rộng là `right_ex - left`, dấu `+ 1` đã "nằm sẵn" trong `right_ex`. Diện tích gọn thành `(right_ex - left) * (bottom_ex - top)` — ít chỗ để quên `+ 1`.

Ví dụ ngoài lề: `first_true(0, 20, lambda v: v * v >= 50)`. Câu hỏi `v*v >= 50` là `False` với `v = 0..7` (`7² = 49 < 50`) và `True` từ `v = 8` (`8² = 64`) → trả `8`. Đó chính là "căn bậc hai của 50, làm tròn lên". Binary search dùng được cho **mọi** predicate đơn điệu, không chỉ cho mảng hay cho ảnh.

In [23]:
def first_true(lo, hi, pred):
    while lo < hi:
        mid = (lo + hi) // 2
        if pred(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


print(first_true(0, 20, lambda v: v * v >= 50))     # 8
print(first_true(0, 10, lambda v: v >= 100))        # 10: không có True -> trả hi
print(first_true(5, 5, lambda v: True))             # 5: vùng rỗng -> trả ngay lo (= hi)


def min_area_generic(image, x, y):
    m, n = len(image), len(image[0])
    left = first_true(0, y, lambda c: has_black_in_col(image, c))
    right_ex = first_true(y + 1, n, lambda c: not has_black_in_col(image, c))
    top = first_true(0, x, lambda r: has_black_in_row(image, r))
    bottom_ex = first_true(x + 1, m, lambda r: not has_black_in_row(image, r))
    return (right_ex - left) * (bottom_ex - top)


print("Example 1:", min_area_generic(image1, x1, y1), "(mong đợi 6)")
print("Example 2:", min_area_generic(image2, x2, y2), "(mong đợi 1)")
print("image_big:", min_area_generic(image_big, x_big, y_big), "(mong đợi 28)")

8
10
5
Example 1: 6 (mong đợi 6)
Example 2: 1 (mong đợi 1)
image_big: 28 (mong đợi 28)


## 14. Code nộp LeetCode

Bản nộp dùng cách 4 (gọn nhất, `O(m log n + n log m)` thời gian, `O(1)` bộ nhớ), với các hàm phụ viết **lồng bên trong** `minArea`. Vài lưu ý:

- Tên method phải là `minArea`, tham số `self, image: List[List[str]], x: int, y: int`, trả về `int`. Trên LeetCode, `List` đã được import sẵn; trong notebook ta đã `from typing import List` ở đầu.
- Hai hàm `col_black`, `row_black` viết lồng bên trong nên "nhìn thấy" biến `image` (closure) — không cần truyền vào.
- Không sửa `image` (tốt: không phá dữ liệu của người gọi), không `print` trong code nộp.

Sau đó ta test bằng `assert` trên 2 ví dụ của đề và các trường hợp biên: ảnh 1 hàng, 1 cột, toàn đen, vùng đen chạm cả 4 mép, `(x, y)` ở góc, hình chữ L, hình "rắn" uốn khúc, một pixel lẻ giữa ảnh.

Kèm theo là bản `SolutionBFS` (cách 2) — cũng được Accepted, hữu ích khi bạn cần một lời giải "chắc ăn" viết nhanh, nhưng **không** đạt yêu cầu "nhanh hơn `O(mn)`" trong trường hợp xấu nhất.

In [24]:
class Solution:
    def minArea(self, image: List[List[str]], x: int, y: int) -> int:
        m, n = len(image), len(image[0])

        def col_black(c):
            return any(row[c] == '1' for row in image)

        def row_black(r):
            return '1' in image[r]

        def first_true(lo, hi, pred):
            while lo < hi:
                mid = (lo + hi) // 2
                if pred(mid):
                    hi = mid
                else:
                    lo = mid + 1
            return lo

        left = first_true(0, y, col_black)
        right_ex = first_true(y + 1, n, lambda c: not col_black(c))
        top = first_true(0, x, row_black)
        bottom_ex = first_true(x + 1, m, lambda r: not row_black(r))
        return (right_ex - left) * (bottom_ex - top)


sol = Solution()
assert sol.minArea([["0","0","1","0"],["0","1","1","0"],["0","1","0","0"]], 0, 2) == 6   # Example 1
assert sol.minArea([["1"]], 0, 0) == 1                                                   # Example 2

edge_cases = [
    # (mô tả, ảnh dạng '#.', x, y, đáp án mong đợi)
    ("1 hàng", ["..###."], 0, 3, 3),
    ("1 cột", [".", "#", "#", "."], 2, 0, 2),
    ("toàn đen 3x4", ["####", "####", "####"], 1, 2, 12),
    ("chạm 4 mép", ["#..", "###", "..#"], 1, 1, 9),
    ("(x, y) ở góc", ["#...", "#...", "####"], 0, 0, 12),
    ("chữ L", ["#..", "#..", "###"], 2, 2, 9),
    ("rắn uốn khúc", ["#####", "....#", "#####", "#....", "#####"], 4, 4, 25),
    ("1 pixel giữa ảnh", [".....", "..#..", "....."], 1, 2, 1),
]
for desc, rows, x, y, expected in edge_cases:
    img = parse_image(rows)
    got = sol.minArea(img, x, y)
    print(f"{desc:<18} -> {got:>3} (mong đợi {expected})")
    assert got == expected and got == min_area_scan(img, x, y)
print("class Solution: tất cả test PASS!")

1 hàng             ->   3 (mong đợi 3)
1 cột              ->   2 (mong đợi 2)
toàn đen 3x4       ->  12 (mong đợi 12)
chạm 4 mép         ->   9 (mong đợi 9)
(x, y) ở góc       ->  12 (mong đợi 12)
chữ L              ->   9 (mong đợi 9)
rắn uốn khúc       ->  25 (mong đợi 25)
1 pixel giữa ảnh   ->   1 (mong đợi 1)
class Solution: tất cả test PASS!


In [25]:
class SolutionBFS:
    def minArea(self, image: List[List[str]], x: int, y: int) -> int:
        from collections import deque
        m, n = len(image), len(image[0])
        top = bottom = x
        left = right = y
        seen = {(x, y)}
        q = deque([(x, y)])
        while q:
            r, c = q.popleft()
            top, bottom = min(top, r), max(bottom, r)
            left, right = min(left, c), max(right, c)
            for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
                if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == '1' and (nr, nc) not in seen:
                    seen.add((nr, nc))
                    q.append((nr, nc))
        return (bottom - top + 1) * (right - left + 1)


sol_bfs = SolutionBFS()
assert sol_bfs.minArea([["0","0","1","0"],["0","1","1","0"],["0","1","0","0"]], 0, 2) == 6
assert sol_bfs.minArea([["1"]], 0, 0) == 1
for desc, rows, x, y, expected in edge_cases:
    assert sol_bfs.minArea(parse_image(rows), x, y) == expected, desc
print("class SolutionBFS: tất cả test PASS!")

class SolutionBFS: tất cả test PASS!


### 14.1. Kiểm thử đối chiếu chéo trên ảnh ngẫu nhiên liên thông

Test viết tay dễ bỏ sót trường hợp. Ta sinh **hàng nghìn** ảnh ngẫu nhiên rồi so mọi cách với **trọng tài** cách 1. Khó khăn: ảnh phải **liên thông** (đúng điều kiện đề). Cách sinh đơn giản — **random walk** (bước đi ngẫu nhiên) của một "con kiến":

1. Chọn ngẫu nhiên một ô, tô đen, đặt con kiến ở đó.
2. Lặp `steps` lần: thỉnh thoảng (xác suất 20%) con kiến "nhảy" về một ô đen **đã tô** bất kỳ (để tạo nhánh rẽ, hình dạng phong phú hơn); còn lại thì nó bước sang một hàng xóm 4 hướng ngẫu nhiên (nếu còn trong lưới) và tô đen ô đó.
3. Chọn `(x, y)` là một ô đen ngẫu nhiên.

**Vì sao chắc chắn liên thông?** Con kiến luôn đứng trên một ô đen. Mỗi ô mới được tô đều **kề** với ô con kiến vừa đứng — mà ô đó đã đen và đã nối với khối. Nên khối đen lớn dần nhưng không bao giờ bị tách rời. (Code vẫn kiểm tra lại cho chắc: BFS từ `(x, y)` phải đi tới đủ mọi ô đen.)

`random.Random(seed)` tạo một bộ sinh số ngẫu nhiên riêng với **seed cố định** → mỗi lần chạy notebook đều ra đúng những ảnh như nhau, nên nếu có lỗi thì tái hiện được.

In [26]:
import random


def random_connected_image(m, n, steps, rng):
    grid = [['0'] * n for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    grid[r][c] = '1'
    painted = [(r, c)]
    for _ in range(steps):
        if rng.random() < 0.2:
            r, c = rng.choice(painted)             # nhảy về một ô đen đã có -> tạo nhánh
            continue
        dr, dc = rng.choice(DIRS)
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            r, c = nr, nc
            if grid[r][c] == '0':
                grid[r][c] = '1'
                painted.append((r, c))
    x, y = rng.choice(painted)
    return grid, x, y


rng = random.Random(302)
sample, sx, sy = random_connected_image(7, 12, 30, rng)
show_grid(sample, start=(sx, sy), title=f"Một ảnh ngẫu nhiên 7 x 12, (x, y) = ({sx}, {sy}):")
print("scan =", min_area_scan(sample, sx, sy), "| bfs =", min_area_bfs(sample, sx, sy),
      "| binary =", min_area_binary(sample, sx, sy), "| generic =", min_area_generic(sample, sx, sy))
print("liên thông (BFS đi tới đủ mọi ô đen)?",
      len(bfs_levels(sample, sx, sy)) == sum(row.count('1') for row in sample))

Một ảnh ngẫu nhiên 7 x 12, (x, y) = (6, 11):
     0 1 2 3 4 5 6 7 8 9 0 1
  0  . . . . . . . . . . . .
  1  . . . . . . . . . . # #
  2  . . . . . . . . . . # #
  3  . . . . . . . . . . . #
  4  . . . . . . . . # # . #
  5  . . . . . . . # # # # #
  6  . . . . . . . # . . . @
scan = 30 | bfs = 30 | binary = 30 | generic = 30
liên thông (BFS đi tới đủ mọi ô đen)? True


In [27]:
rng = random.Random(2024)
methods = {
    "bfs": min_area_bfs,
    "dfs": min_area_dfs,
    "binary": min_area_binary,
    "generic": min_area_generic,
    "Solution": lambda img, x, y: Solution().minArea(img, x, y),
    "SolutionBFS": lambda img, x, y: SolutionBFS().minArea(img, x, y),
}
tests = 0
for _ in range(3000):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    img, x, y = random_connected_image(m, n, rng.randint(0, 60), rng)
    assert len(bfs_levels(img, x, y)) == sum(row.count('1') for row in img)   # đúng là liên thông
    expected = min_area_scan(img, x, y)
    for name, f in methods.items():
        assert f(img, x, y) == expected, (name, img, x, y)
    tests += 1

for _ in range(30):      # vài ảnh kích thước tối đa của đề: 100 x 100
    img, x, y = random_connected_image(100, 100, rng.randint(100, 20000), rng)
    expected = min_area_scan(img, x, y)
    for name, f in methods.items():
        assert f(img, x, y) == expected, name
    tests += 1
print(f"{tests} ảnh ngẫu nhiên liên thông: cả {len(methods)} cách đều khớp với trọng tài (cách 1). PASS!")

3030 ảnh ngẫu nhiên liên thông: cả 6 cách đều khớp với trọng tài (cách 1). PASS!


## 15. Vì sao cách 3 / cách 4 đúng? (invariant bằng lời)

Lời giải đứng trên ba mắt xích:

1. **Bounding box tách theo trục.** `left / right` chỉ phụ thuộc vào tập cột có ô đen; `top / bottom` chỉ phụ thuộc vào tập hàng có ô đen (mục 3, mục 8).
2. **Liên thông ⇒ hình chiếu là một đoạn liền chứa `y` (và `x`).** Đường đi đen đổi cột từng nấc một nên không thể "nhảy qua" cột nào (mục 8.1). Do đó trên `[0, y)` câu hỏi "cột có đen?" có dạng `F..F T..T`; trên `[y+1, n)` câu hỏi "cột trắng?" cũng có dạng `F..F T..T` — đều **đơn điệu**. Tương tự cho hàng.
3. **Binary search đúng trên predicate đơn điệu.** Bất biến của `first_true`: *mọi vị trí `< lo` đều `False`; vị trí `hi` là `True` hoặc là lính canh*.
   - Ban đầu đúng: chưa có vị trí nào `< lo` được xét; `hi` là `y` (cột đen, `True`) hoặc là `n` (lính canh).
   - Mỗi vòng: nếu `pred(mid)` đúng thì đặt `hi = mid` vẫn giữ "vị trí `hi` là `True`"; nếu sai thì vì đơn điệu, mọi vị trí `<= mid` đều sai, đặt `lo = mid + 1` vẫn giữ "mọi vị trí `< lo` sai".
   - Vì `lo <= mid < hi`, vùng `[lo, hi]` co lại ít nhất 1 mỗi vòng → vòng lặp chắc chắn dừng. Khi `lo == hi`, bất biến nói `lo` là `True` đầu tiên.

Ghép lại: `left` là cột đen đầu tiên, `right_ex` là cột trắng đầu tiên sau đoạn đen (= `right + 1`), tương tự cho hàng ⇒ `(right_ex - left) * (bottom_ex - top)` đúng bằng diện tích bounding box.

Còn **cách 2** đúng vì BFS thăm **đúng** thành phần liên thông chứa `(x, y)` (mỗi ô một lần nhờ `visited`), mà đề bảo đảm chỉ có một thành phần — tức là thăm mọi ô đen. **Cách 1** đúng theo định nghĩa, không cần giả thiết gì.

## 16. Độ phức tạp và so sánh các cách

Ký hiệu: `m` = số hàng, `n` = số cột, `B` = số pixel đen (`1 <= B <= mn`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Cần liên thông? | Ghi chú |
|---|---|---|---|---|---|
| 1. Quét toàn bộ | min/max hàng, cột của mọi ô đen | `O(mn)` | `O(1)` | không | dễ nhất, làm trọng tài; đúng cả khi ảnh nhiều vùng |
| 2. BFS / DFS từ `(x, y)` | đi hết vùng đen, cập nhật 4 biên | `O(B)` | `O(B)` | có (để không sót) | rất nhanh khi vùng nhỏ; xấu nhất `B = mn` |
| 3. Binary search 4 lần | hình chiếu đơn điệu, tìm 4 chỗ đổi | `O(m log n + n log m)` | `O(1)` | có (để đơn điệu) | **cách chính**, đạt yêu cầu đề |
| 4. `first_true` tổng quát | như cách 3, một hàm cho 4 biên | `O(m log n + n log m)` | `O(1)` | có | gọn, ít chỗ sai, dùng để nộp |

Bảng đếm số lần đọc ở mục 11 cho thấy một điều thú vị: với vùng đen **rất nhỏ** (ô vuông 2×2), cách 2 đọc ít ô hơn hẳn cách 3; còn với vùng đen **rất lớn** (toàn đen), cách 2 lại tệ nhất trong ba cách, trong khi cách 3 chỉ đọc đúng 12 ô (mỗi câu hỏi gặp ô đen ngay ở vị trí đầu tiên). Không có cách nào "luôn đọc ít nhất". Cách 3 được chọn vì nó tốt trong **trường hợp xấu nhất** — đúng thứ đề yêu cầu ("less than `O(mn)`" cho mọi input).

## 17. Lỗi người mới hay gặp

### Lỗi 1: So sánh với số `1` thay vì chuỗi `'1'`

```python
if image[r][c] == 1:      # SAI: '1' == 1 là False -> không bao giờ thấy ô đen
```

**Sửa:** `image[r][c] == '1'`.

### Lỗi 2: Nhầm `x` là cột, `y` là hàng

Đề nói `image[x][y] == '1'` → `x` là **hàng**. Đổi chỗ `(y, x)` khi gọi sẽ tìm sai vùng (thậm chí `IndexError` khi `m != n`). **Sửa:** biên cột (`left/right`) dùng `y`; biên hàng (`top/bottom`) dùng `x`.

### Lỗi 3: Quên `+ 1` khi tính diện tích từ biên "bao gồm"

`(right - left) * (bottom - top)` cho Example 1 ra `1 * 2 = 2`. **Sửa:** `(right - left + 1) * (bottom - top + 1)`, hoặc dùng biên exclusive như cách 4.

### Lỗi 4: Tìm biên trái trên **cả** `[0, n)` thay vì `[0, y]`

Cả hình chiếu có dạng `F..F T..T F..F` — **không** đơn điệu. Nếu `mid` rơi vào phần `F` bên phải, code tưởng "chưa tới vùng đen" và chạy tiếp sang phải → sai. **Sửa:** chỉ tìm trong `[0, y]` (và "cột trắng đầu tiên" trong `[y+1, n)` cho biên phải).

### Lỗi 5: Tìm "cột đen cuối cùng" bằng `lo = mid` với `mid` làm tròn xuống → lặp vô hạn

```python
lo, hi = y, n - 1
while lo < hi:
    mid = (lo + hi) // 2
    if has_black_in_col(image, mid): lo = mid      # khi hi = lo + 1: mid = lo -> đứng yên!
    else:                            hi = mid - 1
```

**Sửa:** hoặc làm tròn **lên** `mid = (lo + hi + 1) // 2`, hoặc (như notebook này) đổi câu hỏi thành "cột trắng đầu tiên" để chỉ dùng đúng một mẫu.

### Lỗi 6 (BFS): Đánh dấu `visited` lúc lấy ra thay vì lúc cho vào

Một ô có thể bị nhiều hàng xóm cùng cho vào hàng đợi trước khi nó được lấy ra → hàng đợi phình to, làm việc thừa (vẫn ra đúng nhưng chậm). Quên hẳn `visited` thì còn tệ hơn: **lặp vô hạn**. **Sửa:** `visited.add(...)` ngay trước `append`.

### Lỗi 7 (DFS): Đệ quy quá sâu

DFS đệ quy trên ảnh `100 × 100` toàn đen có thể sâu hàng nghìn tầng → `RecursionError`. **Sửa:** dùng stack tự quản (mục 7.1) hoặc BFS.

Hai cell dưới chạy thử từng lỗi (các vòng lặp có giới hạn số bước để không treo máy).

In [28]:
# Lỗi 1: so với số 1
count_int = sum(1 for row in image1 for v in row if v == 1)
count_str = sum(1 for row in image1 for v in row if v == '1')
print("Lỗi 1: số ô đen đếm bằng == 1:", count_int, "| đếm bằng == '1':", count_str)

# Lỗi 2: đổi chỗ x và y
print("Lỗi 2: min_area_binary(image1, y, x) =", min_area_binary(image1, y1, x1), "(đúng là 6)")

# Lỗi 3: quên + 1
L, R = find_left(image1, y1), find_right(image1, y1)
T, B = find_top(image1, x1), find_bottom(image1, x1)
print("Lỗi 3: quên +1 ->", (R - L) * (B - T), "| đúng ->", (R - L + 1) * (B - T + 1))

# Lỗi 4: tìm biên trái trên cả [0, n)
img4 = parse_image(["##......", "#.......", "##......"])
wrong_left = first_true(0, len(img4[0]), lambda c: has_black_in_col(img4, c))
print("Lỗi 4: hình chiếu cột =", col_projection(img4))
print("       tìm trên cả [0, n) ->", wrong_left, "| tìm trên [0, y] với y = 1 ->", find_left(img4, 1))


# Lỗi 5: "cột đen cuối cùng" với mid làm tròn xuống
def find_right_buggy(image, y, max_loops=20):
    lo, hi, loops = y, len(image[0]) - 1, 0
    while lo < hi:
        loops += 1
        if loops > max_loops:
            return f"LẶP VÔ HẠN (lo={lo}, hi={hi} không đổi)"
        mid = (lo + hi) // 2
        if has_black_in_col(image, mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


print("Lỗi 5:", find_right_buggy(image1, y1), "| bản đúng: right =", find_right(image1, y1))

Lỗi 1: số ô đen đếm bằng == 1: 0 | đếm bằng == '1': 4
Lỗi 2: min_area_binary(image1, y, x) = 9 (đúng là 6)
Lỗi 3: quên +1 -> 2 | đúng -> 6
Lỗi 4: hình chiếu cột = [1, 1, 0, 0, 0, 0, 0, 0]
       tìm trên cả [0, n) -> 8 | tìm trên [0, y] với y = 1 -> 0
Lỗi 5: LẶP VÔ HẠN (lo=2, hi=3 không đổi) | bản đúng: right = 2


In [29]:
import sys


def bfs_count_pushes(image, x, y, mark_on_push=True):
    """Đếm số lần append vào hàng đợi, với hai cách đánh dấu visited."""
    m, n = len(image), len(image[0])
    visited = {(x, y)} if mark_on_push else set()
    q = deque([(x, y)])
    pushes = 1
    while q:
        r, c = q.popleft()
        if not mark_on_push:                  # đánh dấu MUỘN: lúc lấy ra
            if (r, c) in visited:
                continue
            visited.add((r, c))
        for nr, nc in neighbors(r, c, m, n):
            if image[nr][nc] == '1' and (nr, nc) not in visited:
                if mark_on_push:              # đánh dấu SỚM: lúc cho vào
                    visited.add((nr, nc))
                q.append((nr, nc))
                pushes += 1
    return pushes


full30 = [['1'] * 30 for _ in range(30)]
print("Lỗi 6: ảnh 30x30 toàn đen (900 ô):")
print("   đánh dấu lúc cho vào:", bfs_count_pushes(full30, 0, 0, True), "lần append")
print("   đánh dấu lúc lấy ra :", bfs_count_pushes(full30, 0, 0, False), "lần append")


def min_area_dfs_recursive(image, x, y):
    m, n = len(image), len(image[0])
    box = [x, x, y, y]                        # top, bottom, left, right
    visited = set()

    def dfs(r, c):
        if not (0 <= r < m and 0 <= c < n) or image[r][c] != '1' or (r, c) in visited:
            return
        visited.add((r, c))
        box[0], box[1] = min(box[0], r), max(box[1], r)
        box[2], box[3] = min(box[2], c), max(box[3], c)
        for dr, dc in DIRS:
            dfs(r + dr, c + dc)               # hàm tự gọi lại chính nó

    dfs(x, y)
    return (box[1] - box[0] + 1) * (box[3] - box[2] + 1)


full100 = [['1'] * 100 for _ in range(100)]
print("Lỗi 7: giới hạn đệ quy hiện tại =", sys.getrecursionlimit())
print("   DFS đệ quy trên Example 1:", min_area_dfs_recursive(image1, x1, y1))
try:
    print("   DFS đệ quy trên 100x100 toàn đen:", min_area_dfs_recursive(full100, 0, 0))
except RecursionError as e:
    print("   DFS đệ quy trên 100x100 toàn đen: RecursionError -", e)
print("   DFS bằng stack trên 100x100 toàn đen:", min_area_dfs(full100, 0, 0))

Lỗi 6: ảnh 30x30 toàn đen (900 ô):
   đánh dấu lúc cho vào: 900 lần append
   đánh dấu lúc lấy ra : 1741 lần append
Lỗi 7: giới hạn đệ quy hiện tại = 1000
   DFS đệ quy trên Example 1: 6
   DFS đệ quy trên 100x100 toàn đen: RecursionError - maximum recursion depth exceeded
   DFS bằng stack trên 100x100 toàn đen: 10000


## 18. Mẹo nhận dạng pattern

- Đề hỏi **"hình chữ nhật bao" / bounding box** → nhớ rằng nó tách thành **min/max theo từng trục độc lập** → bài 2D thành hai bài 1D trên **hình chiếu**.
- Đề nhấn mạnh **"connected" / "only one component"** mà lại đòi **nhanh hơn duyệt hết** → nghĩ tới tính chất "hình chiếu là đoạn liền" ⇒ **đơn điệu** ⇒ **binary search**.
- Đề cho sẵn **một điểm nằm trong vùng** `(x, y)` → đó thường là **chỗ cắt** chia dãy thành hai nửa đơn điệu, đồng thời là vị trí "chắc chắn đúng" để làm `hi` cho lần tìm đầu tiên.
- **Binary search không chỉ dành cho mảng sắp xếp**: bất cứ câu hỏi Có/Không nào trên chỉ số mà câu trả lời có dạng `F..F T..T` đều chia đôi được ("binary search on a predicate"). Mỗi câu hỏi có thể tốn `O(m)` — vẫn lời nếu chỉ hỏi `O(log n)` lần.
- Muốn tìm vị trí **cuối cùng** thỏa điều kiện mà ngại mẫu làm tròn lên → **đổi câu hỏi** sang phủ định, tìm vị trí **đầu tiên** thỏa câu hỏi mới rồi trừ 1.
- Cần **đi hết một vùng** trên lưới (đếm ô, tô màu, tính diện tích vùng, đếm số đảo) → **BFS/DFS + visited** (flood fill).
- Bộ test nên luôn có: ảnh `1 × 1`, 1 hàng, 1 cột, toàn đen, vùng chạm mép, `(x, y)` ở góc hoặc ở mép vùng, hình uốn khúc.

## 19. Biến thể và bài liên quan

| Bài | Nội dung | Liên hệ với bài này |
|---|---|---|
| 200. Number of Islands | đếm số vùng `'1'` | BFS/DFS + visited như cách 2, chạy từ mọi ô đen chưa thăm (bài tập 6) |
| 695. Max Area of Island | diện tích (số ô) của vùng lớn nhất | BFS đếm số ô thay vì cập nhật 4 biên |
| 733. Flood Fill | tô màu vùng chứa một ô cho trước | đúng "vết dầu loang" ở mục 6 |
| 278. First Bad Version | tìm phiên bản lỗi đầu tiên | đúng mẫu `first_true` với predicate `isBadVersion` |
| 35. Search Insert Position | vị trí chèn vào mảng sắp xếp | `first_true(0, n, lambda i: nums[i] >= target)` |
| 34. Find First and Last Position of Element in Sorted Array | đầu và cuối của một đoạn | giống tìm `left` và `right` ở đây |
| 74. Search a 2D Matrix | tìm một giá trị trong ma trận | binary search trên lưới 2D |

Cell dưới dùng lại **nguyên** hàm `first_true` cho hai bài 278 và 35 — học một mẫu tổng quát thì dùng được ở nhiều bài.

In [30]:
# LeetCode 278 — First Bad Version: mô phỏng API isBadVersion (phiên bản từ 37 trở đi bị lỗi)
first_bad = 37
isBadVersion = lambda v: v >= first_bad
n_versions = 100
print("278 -> phiên bản lỗi đầu tiên:", first_true(1, n_versions + 1, isBadVersion))

# LeetCode 35 — Search Insert Position: vị trí đầu tiên có nums[i] >= target
nums = [1, 3, 5, 6]
for target in [5, 2, 7, 0]:
    pos = first_true(0, len(nums), lambda i: nums[i] >= target)
    print(f"35 -> chèn {target} vào {nums} ở vị trí {pos}")

278 -> phiên bản lỗi đầu tiên: 37
35 -> chèn 5 vào [1, 3, 5, 6] ở vị trí 2
35 -> chèn 2 vào [1, 3, 5, 6] ở vị trí 1
35 -> chèn 7 vào [1, 3, 5, 6] ở vị trí 4
35 -> chèn 0 vào [1, 3, 5, 6] ở vị trí 0


## 20. Bài tập vận dụng

Làm theo thứ tự từ dễ tới khó. Gợi ý ở mục 21, đáp án ở mục 22 (cuối notebook). Các cell TODO chỉ **định nghĩa** hàm, chưa gọi — làm xong, bạn bỏ dấu `#` ở các dòng `assert` để tự kiểm tra.

### Bài 1 — Tính tay (không chạy code)

Cho ảnh 5 × 7, `(x, y) = (2, 3)` (ô `@` là ô đen xuất phát):

```text
     0 1 2 3 4 5 6
  0  . . . . . . .
  1  . . # # . . .
  2  . . . @ # # .
  3  . . . # . . .
  4  . . . . . . .
```

- (a) Viết hình chiếu cột và hình chiếu hàng (dãy 0/1).
- (b) Tính `top, bottom, left, right` và diện tích.
- (c) Lập bảng `lo, hi, mid, câu hỏi, trả lời, cập nhật` cho lần tìm `left` của cách 3 (`find_left`, vùng `[0, 3]`).

### Bài 2 — Đoán output

Không chạy code, đoán:

- (a) `first_true(0, 10, lambda v: v * v * v >= 100)`
- (b) `first_true(7, 7, lambda v: v > 100)`
- (c) `first_true(2, 6, lambda v: False)`
- (d) `min_area_binary(parse_image(["##.##"]), 0, 4)` — ảnh này **không** liên thông (vi phạm đề). Cách 3 trả về gì? Còn `min_area_scan` trả về gì? Vì sao khác nhau?

### Bài 3 — Điền TODO: cách 4 với `first_true`

Hoàn thành `min_area_todo` bằng cách gọi `first_true` 4 lần. Chú ý vùng tìm của từng biên, và câu hỏi là "đen" hay "trắng".

In [31]:
def min_area_todo(image, x, y):
    m, n = len(image), len(image[0])
    # TODO 1: left = cột ĐEN đầu tiên, tìm trong [0, y)
    # TODO 2: right_ex = cột TRẮNG đầu tiên, tìm trong [y + 1, n)
    # TODO 3: top = hàng ĐEN đầu tiên, tìm trong [0, x)
    # TODO 4: bottom_ex = hàng TRẮNG đầu tiên, tìm trong [x + 1, m)
    # TODO 5: trả về (right_ex - left) * (bottom_ex - top)
    raise NotImplementedError


# Bỏ dấu # sau khi làm xong:
# assert min_area_todo(image1, 0, 2) == 6
# assert min_area_todo(image2, 0, 0) == 1
# assert min_area_todo(image_big, 2, 5) == 28

### Bài 4 — Tự viết lại: BFS (cách 2) không nhìn code mẫu

Viết `min_area_bfs_self(image, x, y)` dùng `deque` và `visited`. Yêu cầu:

- đánh dấu `visited` **lúc cho vào** hàng đợi;
- kiểm tra "còn trong lưới" **trước** khi đọc `image[nr][nc]`;
- khởi tạo 4 biên bằng `(x, y)`.

In [32]:
def min_area_bfs_self(image, x, y):
    # TODO: khởi tạo top/bottom/left/right bằng (x, y)
    # TODO: visited = {(x, y)}, q = deque([(x, y)])
    # TODO: while q: popleft, cập nhật 4 biên, xét 4 hàng xóm theo DIRS
    pass


# assert min_area_bfs_self(image1, 0, 2) == 6
# assert min_area_bfs_self(image2, 0, 0) == 1
# assert min_area_bfs_self(image_big, 2, 5) == 28

### Bài 5 — Biến thể: liên thông 8 hướng

Giả sử đề đổi thành: hai pixel kề nhau nếu chung **cạnh hoặc góc** (8 hướng: thêm 4 hướng chéo). Ví dụ ảnh

```text
#..
.#.
..#
```

bây giờ là **một** vùng (một đường chéo), bounding box `3 × 3 = 9`.

- (a) Viết `min_area_bfs8(image, x, y)` — BFS 8 hướng (dùng `DIRS8` cho sẵn).
- (b) Câu hỏi suy nghĩ: cách 3 (binary search) **còn đúng không** khi liên thông 8 hướng? Giải thích bằng lập luận "đổi cột từng nấc" ở mục 8.1.
- (c) Kiểm chứng: sinh ảnh ngẫu nhiên liên thông 8 hướng (random walk với 8 hướng), so `min_area_bfs8` và `min_area_binary` với trọng tài `min_area_scan`.

In [33]:
DIRS8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


def min_area_bfs8(image, x, y):
    # TODO: giống min_area_bfs nhưng duyệt 8 hướng DIRS8
    pass


# diag = parse_image(["#..", ".#.", "..#"])
# assert min_area_bfs8(diag, 0, 0) == 9
# assert min_area_bfs(diag, 0, 0) == 1      # BFS 4 hướng chỉ thấy đúng 1 ô!

### Bài 6 — Mini challenge: bounding box của MỌI vùng

Bỏ điều kiện "chỉ một vùng": ảnh có thể có nhiều vùng đen (liên thông 4 hướng) hoặc không có vùng nào, và đề **không** cho `(x, y)`. Viết `all_box_areas(image)` trả về **list diện tích bounding box của từng vùng, sắp xếp tăng dần**. Ví dụ:

```text
##..#
##..#
....#
#....
```

có 3 vùng: khối 2×2 (diện tích 4), cột phải dài 3 ô (diện tích 3), ô đơn ở góc dưới trái (diện tích 1) → `[1, 3, 4]`.

Câu hỏi thêm: vì sao ở bài này **không** dùng được binary search của cách 3?

In [34]:
def all_box_areas(image):
    # TODO: duyệt mọi ô; gặp ô đen CHƯA thăm -> BFS đi hết vùng đó, tính bounding box của vùng
    # TODO: gom diện tích từng vùng vào list, sắp xếp rồi trả về
    pass


# assert all_box_areas(parse_image(["##..#", "##..#", "....#", "#...."])) == [1, 3, 4]
# assert all_box_areas(image1) == [6]
# assert all_box_areas(parse_image(["...", "..."])) == []

## 21. Gợi ý theo mức

(Mức 1: hướng tư duy · Mức 2: cấu trúc / cú pháp · Mức 3: gần như code)

**Bài 1**
- Mức 1: hình chiếu cột = "cột này có ít nhất một `#` (hoặc `@`) không?".
- Mức 2: `left / right` = vị trí số 1 đầu tiên / cuối cùng của hình chiếu cột; `top / bottom` tương tự với hình chiếu hàng.
- Mức 3: (c) bắt đầu `lo = 0, hi = 3` → `mid = 1`; cột 1 có `#` không? Rồi cập nhật `lo` hay `hi`?

**Bài 2**
- Mức 1: tự lập dãy `F/T` của predicate trên vùng `[lo, hi)` rồi tìm `T` đầu tiên.
- Mức 2: vùng rỗng (`lo == hi`) thì vòng `while` không chạy lần nào; không có `True` thì trả `hi`.
- Mức 3: (d) hình chiếu cột là `1 1 0 1 1`; nửa trái `[0, 4]` là `1 1 0 1 | 1` — không đơn điệu. Chạy tay `find_left` với `lo = 0, hi = 4`: `mid = 2` trước.

**Bài 3**
- Mức 1: xem lại bảng 4 lời gọi ở mục 13.
- Mức 2: câu hỏi "trắng" là `not has_black_in_col(image, c)`.
- Mức 3: `left = first_true(0, y, lambda c: has_black_in_col(image, c))`; ba dòng còn lại tương tự.

**Bài 4**
- Mức 1: 5 bước ở mục 6.4.
- Mức 2: `r, c = q.popleft()`; `for dr, dc in DIRS:`; `nr, nc = r + dr, c + dc`.
- Mức 3: điều kiện thêm hàng xóm: `0 <= nr < m and 0 <= nc < n and image[nr][nc] == '1' and (nr, nc) not in visited`.

**Bài 5**
- Mức 1: một bước đi chéo làm chỉ số cột thay đổi bao nhiêu?
- Mức 2: chỉ cần đổi `DIRS` thành `DIRS8` — trong BFS và trong hàm sinh ảnh random walk.
- Mức 3: mỗi bước 8 hướng đổi cột **tối đa 1** → vẫn "không nhảy bậc" → hình chiếu vẫn là đoạn liền → `min_area_binary` giữ nguyên.

**Bài 6**
- Mức 1: giống bài 200 Number of Islands, nhưng với mỗi đảo tính thêm bounding box.
- Mức 2: `visited` dùng chung cho cả ảnh; hai vòng `for r ... for c ...` ở ngoài cùng tìm "ô đen chưa thăm" để bắt đầu một BFS mới.
- Mức 3: trong BFS của mỗi đảo, khởi tạo 4 biên bằng ô bắt đầu, cập nhật khi `popleft`, hết hàng đợi thì `areas.append((bottom - top + 1) * (right - left + 1))`.

## 22. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

### Đáp án bài 1

(a) Hình chiếu cột: `0 0 1 1 1 1 0`. Hình chiếu hàng: `0 1 1 1 0`.

(b) `top = 1, bottom = 3, left = 2, right = 5` → diện tích `(3 - 1 + 1) * (5 - 2 + 1) = 3 * 4 = 12`.

(c)

| lo | hi | mid | câu hỏi | trả lời | cập nhật |
|---:|---:|---:|---|---|---|
| 0 | 3 | 1 | cột 1 có đen? | `False` | `lo = 2` |
| 2 | 3 | 2 | cột 2 có đen? | `True` | `hi = 2` → dừng, `left = 2` |

### Đáp án bài 2

- (a) `5` (`4³ = 64 < 100`, `5³ = 125 >= 100`).
- (b) `7` (vùng rỗng, trả ngay `lo`).
- (c) `6` (không có `True` → trả `hi`).
- (d) Cách 3 trả `2`, còn `min_area_scan` trả `5`. Khi tìm `left` trong `[0, 4]`: `mid = 2`, cột 2 trắng → `lo = 3` — binary search "nhảy qua" lỗ hổng và bỏ mất khối bên trái, kết luận `left = 3`. Ảnh không liên thông làm hình chiếu có lỗ → mất tính đơn điệu → binary search sai. Cách 1 không dựa vào liên thông nên vẫn đúng.

Cell dưới kiểm chứng bài 1 và bài 2 (kèm trace đầy đủ của bài 1).

In [35]:
ex1_img = parse_image([".......", "..##...", "...###.", "...#...", "......."])
print("Bài 1 (a): cột", col_projection(ex1_img), "| hàng", row_projection(ex1_img))
assert col_projection(ex1_img) == [0, 0, 1, 1, 1, 1, 0] and row_projection(ex1_img) == [0, 1, 1, 1, 0]
assert min_area_scan(ex1_img, 2, 3) == 12 and min_area_binary(ex1_img, 2, 3) == 12
assert find_left(ex1_img, 3) == 2
trace_min_area_binary(ex1_img, 2, 3)

assert first_true(0, 10, lambda v: v * v * v >= 100) == 5
assert first_true(7, 7, lambda v: v > 100) == 7
assert first_true(2, 6, lambda v: False) == 6
split5 = parse_image(["##.##"])
assert min_area_binary(split5, 0, 4) == 2 and min_area_scan(split5, 0, 4) == 5
print("Bài 1, Bài 2: PASS — (d) binary =", min_area_binary(split5, 0, 4), ", scan =", min_area_scan(split5, 0, 4))

Bài 1 (a): cột [0, 0, 1, 1, 1, 1, 0] | hàng [0, 1, 1, 1, 0]
[left]   cột ĐEN đầu tiên trong [0, 3]  (cột 3 = y, chắc chắn đen)
   lo=0 hi=3 mid=1 | cột 1 có đen?   False -> lo = mid + 1
   lo=2 hi=3 mid=2 | cột 2 có đen?   True  -> hi = mid
   => left = 2
[right]  cột TRẮNG đầu tiên trong [4, 7]  (cột 7 = cột ảo, coi như trắng)
   lo=4 hi=7 mid=5 | cột 5 trắng?    False -> lo = mid + 1
   lo=6 hi=7 mid=6 | cột 6 trắng?    True  -> hi = mid
   => cột trắng đầu tiên = 6, right = 5
[top]    hàng ĐEN đầu tiên trong [0, 2]  (hàng 2 = x, chắc chắn đen)
   lo=0 hi=2 mid=1 | hàng 1 có đen?  True  -> hi = mid
   lo=0 hi=1 mid=0 | hàng 0 có đen?  False -> lo = mid + 1
   => top = 1
[bottom] hàng TRẮNG đầu tiên trong [3, 5]  (hàng 5 = hàng ảo, coi như trắng)
   lo=3 hi=5 mid=4 | hàng 4 trắng?   True  -> hi = mid
   lo=3 hi=4 mid=3 | hàng 3 trắng?   False -> lo = mid + 1
   => hàng trắng đầu tiên = 4, bottom = 3
diện tích = (5 - 2 + 1) * (3 - 1 + 1) = 12
Bài 1, Bài 2: PASS — (d) binary = 2 , scan 

In [36]:
def answer_min_area_todo(image, x, y):
    m, n = len(image), len(image[0])
    left = first_true(0, y, lambda c: has_black_in_col(image, c))
    right_ex = first_true(y + 1, n, lambda c: not has_black_in_col(image, c))
    top = first_true(0, x, lambda r: has_black_in_row(image, r))
    bottom_ex = first_true(x + 1, m, lambda r: not has_black_in_row(image, r))
    return (right_ex - left) * (bottom_ex - top)


def answer_min_area_bfs_self(image, x, y):
    m, n = len(image), len(image[0])
    top = bottom = x
    left = right = y
    visited = {(x, y)}
    q = deque([(x, y)])
    while q:
        r, c = q.popleft()
        top, bottom = min(top, r), max(bottom, r)
        left, right = min(left, c), max(right, c)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == '1' and (nr, nc) not in visited:
                visited.add((nr, nc))
                q.append((nr, nc))
    return (bottom - top + 1) * (right - left + 1)


for f in (answer_min_area_todo, answer_min_area_bfs_self):
    assert f(image1, 0, 2) == 6
    assert f(image2, 0, 0) == 1
    assert f(image_big, 2, 5) == 28

rng = random.Random(33)
for _ in range(500):
    img, x, y = random_connected_image(rng.randint(1, 10), rng.randint(1, 10), rng.randint(0, 40), rng)
    expected = min_area_scan(img, x, y)
    assert answer_min_area_todo(img, x, y) == expected == answer_min_area_bfs_self(img, x, y)
print("Bài 3, Bài 4: PASS (ví dụ của đề, image_big + 500 ảnh ngẫu nhiên)")

Bài 3, Bài 4: PASS (ví dụ của đề, image_big + 500 ảnh ngẫu nhiên)


### Đáp án bài 5 (b) và câu hỏi thêm của bài 6

**Bài 5 (b):** Vẫn đúng! Một bước đi 8 hướng làm chỉ số cột thay đổi **tối đa 1** (và chỉ số hàng cũng thay đổi tối đa 1). Vậy đường đi đen từ cột `L` sang cột `R` vẫn phải ghé **mọi** cột ở giữa → hình chiếu vẫn là đoạn liền chứa `y` → mỗi nửa vẫn đơn điệu. Chỉ BFS mới phải đổi sang 8 hướng; `min_area_binary` giữ nguyên không sửa một dòng nào. (Ngược lại, BFS 4 hướng sẽ **sai** trên ảnh 8 hướng: với đường chéo nó chỉ thấy đúng 1 ô.)

**Bài 6:** Binary search dựa vào "hình chiếu là **một** đoạn liền", mà nhiều vùng thì hình chiếu có thể có lỗ (như `image_split`). Quan trọng hơn, ta cần box của **từng** vùng riêng, trong khi hình chiếu "trộn" mọi vùng lại thành một cái bóng chung — không thể tách ra được. Phải đi hết ảnh: vòng ngoài tìm ô đen chưa thăm, BFS bên trong đi hết vùng đó. Mỗi ô vẫn chỉ được xử lý một lần nên tổng thời gian `O(mn)`.

In [37]:
def answer_min_area_bfs8(image, x, y):
    m, n = len(image), len(image[0])
    top = bottom = x
    left = right = y
    visited = {(x, y)}
    q = deque([(x, y)])
    while q:
        r, c = q.popleft()
        top, bottom = min(top, r), max(bottom, r)
        left, right = min(left, c), max(right, c)
        for dr, dc in DIRS8:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and image[nr][nc] == '1' and (nr, nc) not in visited:
                visited.add((nr, nc))
                q.append((nr, nc))
    return (bottom - top + 1) * (right - left + 1)


def random_connected_image8(m, n, steps, rng):
    """Như random_connected_image nhưng con kiến bước 8 hướng -> ảnh liên thông 8 hướng."""
    grid = [['0'] * n for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    grid[r][c] = '1'
    painted = [(r, c)]
    for _ in range(steps):
        if rng.random() < 0.2:
            r, c = rng.choice(painted)
            continue
        dr, dc = rng.choice(DIRS8)
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            r, c = nr, nc
            if grid[r][c] == '0':
                grid[r][c] = '1'
                painted.append((r, c))
    x, y = rng.choice(painted)
    return grid, x, y


diag = parse_image(["#..", ".#.", "..#"])
assert answer_min_area_bfs8(diag, 0, 0) == 9 and min_area_bfs(diag, 0, 0) == 1
assert min_area_binary(diag, 0, 0) == 9                # binary search vẫn đúng với 8 hướng
rng = random.Random(88)
for _ in range(1000):
    img, x, y = random_connected_image8(rng.randint(1, 10), rng.randint(1, 10), rng.randint(0, 40), rng)
    expected = min_area_scan(img, x, y)
    assert answer_min_area_bfs8(img, x, y) == expected
    assert min_area_binary(img, x, y) == expected
print("Bài 5: BFS 8 hướng và binary search đều khớp trọng tài trên 1000 ảnh liên thông 8 hướng — PASS")


def answer_all_box_areas(image):
    m, n = len(image), len(image[0])
    visited = set()
    areas = []
    for sr in range(m):
        for sc in range(n):
            if image[sr][sc] != '1' or (sr, sc) in visited:
                continue
            top = bottom = sr                 # bắt đầu một đảo mới tại (sr, sc)
            left = right = sc
            visited.add((sr, sc))
            q = deque([(sr, sc)])
            while q:
                r, c = q.popleft()
                top, bottom = min(top, r), max(bottom, r)
                left, right = min(left, c), max(right, c)
                for nr, nc in neighbors(r, c, m, n):
                    if image[nr][nc] == '1' and (nr, nc) not in visited:
                        visited.add((nr, nc))
                        q.append((nr, nc))
            areas.append((bottom - top + 1) * (right - left + 1))
    return sorted(areas)


assert answer_all_box_areas(parse_image(["##..#", "##..#", "....#", "#...."])) == [1, 3, 4]
assert answer_all_box_areas(image1) == [6]
assert answer_all_box_areas(parse_image(["...", "..."])) == []
assert answer_all_box_areas(image_split) == [2, 4]
print("Bài 6: PASS —", answer_all_box_areas(parse_image(["##..#", "##..#", "....#", "#...."])),
      "| image_split ->", answer_all_box_areas(image_split))

Bài 5: BFS 8 hướng và binary search đều khớp trọng tài trên 1000 ảnh liên thông 8 hướng — PASS
Bài 6: PASS — [1, 3, 4] | image_split -> [2, 4]


## 23. Template ngắn nên nhớ

```python
class Solution:
    def minArea(self, image: List[List[str]], x: int, y: int) -> int:
        m, n = len(image), len(image[0])
        col_black = lambda c: any(row[c] == '1' for row in image)   # O(m)
        row_black = lambda r: '1' in image[r]                       # O(n)

        def first_true(lo, hi, pred):      # F..F T..T trên [lo, hi) -> vị trí T đầu tiên (không có: hi)
            while lo < hi:
                mid = (lo + hi) // 2
                if pred(mid): hi = mid
                else:         lo = mid + 1
            return lo

        left      = first_true(0, y, col_black)                        # cột đen đầu tiên
        right_ex  = first_true(y + 1, n, lambda c: not col_black(c))   # cột trắng đầu tiên bên phải
        top       = first_true(0, x, row_black)                        # hàng đen đầu tiên
        bottom_ex = first_true(x + 1, m, lambda r: not row_black(r))   # hàng trắng đầu tiên bên dưới
        return (right_ex - left) * (bottom_ex - top)
```

Câu thần chú: **"Liên thông ⇒ hình chiếu liền ⇒ cắt tại `(x, y)` được hai nửa đơn điệu ⇒ 4 lần binary search."**

## 24. Checklist trước khi submit

- So với chuỗi `'1'`, không phải số `1`?
- `x` là **hàng** (dùng cho `top / bottom`, `row_black`), `y` là **cột** (dùng cho `left / right`, `col_black`)?
- `m = len(image)`, `n = len(image[0])` — không đảo chỗ?
- Biên trái tìm trong `[0, y)`, biên phải tìm "cột **trắng** đầu tiên" trong `[y + 1, n)`; tương tự cho hàng — **không** tìm trên cả dãy?
- Mẫu binary search đúng: `while lo < hi`, `mid = (lo + hi) // 2`, `hi = mid` / `lo = mid + 1`; không bao giờ đọc vị trí `hi`?
- Diện tích: biên "bao gồm" thì `+ 1`; biên exclusive (`right_ex`, `bottom_ex`) thì **không** `+ 1` — không trộn hai kiểu?
- Đã test: 2 ví dụ của đề, ảnh `1 × 1`, 1 hàng, 1 cột, toàn đen, vùng chạm mép, `(x, y)` ở góc, hình uốn khúc?
- (Nếu dùng BFS) đánh dấu `visited` lúc cho vào hàng đợi, kiểm tra trong lưới trước khi đọc ô, không đệ quy quá sâu?
- Trả lời được câu "vì sao nhanh hơn `O(mn)`?": `O(m log n + n log m)`, nhờ **tính liên thông** làm hình chiếu đơn điệu?